
# Theme 9 : Reseau de neurones from scratch en NumPy
#  Etudiant : Frederic BALAGUEMA | MI3 FAST Natitingou

In [ ]:
import numpy as np           # calcul matriciel (dot, exp, log, etc.)
import pandas as pd
import matplotlib.pyplot as plt  # tracer les courbes de loss
import matplotlib.patches as mpatches
import time                  # mesurer le temps d'entrainement
from sklearn.datasets import load_iris
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')
# Keras est inclus dans TensorFlow
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# Fixer la graine aleatoire pour avoir des resultats reproductibles
# (meme valeurs aleatoires a chaque execution)
np.random.seed(42)

print("Imports OK")

In [ ]:
#Fonctions d'activation et leurs derivees
def sigmoid(Z):
    """
    Fonction sigmoide : ecrase Z dans l'intervalle ]0, 1[
    Utilisee en couche de sortie pour classification binaire.

    Parametre :
        Z : np.ndarray de n'importe quelle shape

    Retourne :
        meme shape que Z, valeurs comprises entre 0 et 1
    """
    # np.clip evite le overflow numerique :
    # e^(-500) est trop petit pour Python, e^(500) est infini
    # on borne Z entre -500 et 500 avant le calcul
    Z_clip = np.clip(Z, -500, 500)

    # formule : 1 / (1 + e^(-z))
    return 1 / (1 + np.exp(-Z_clip))


def sigmoid_derivative(Z):
    """
    Derivee de la sigmoide : sigma'(z) = sigma(z) * (1 - sigma(z))
    Utilisee pendant la backpropagation.

    Parametre :
        Z : np.ndarray (valeurs AVANT activation, pas apres)

    Retourne :
        meme shape que Z
    """
    s = sigmoid(Z)           # on calcule d'abord sigma(z)
    return s * (1 - s)       # puis on applique la formule de la derivee


def relu(Z):
    """
    Fonction ReLU : max(0, z)
    Utilisee dans les couches cachees.

    Parametre :
        Z : np.ndarray de n'importe quelle shape

    Retourne :
        meme shape que Z, valeurs >= 0
    """
    # np.maximum compare element par element Z et 0
    # si Z[i,j] > 0 : retourne Z[i,j]
    # si Z[i,j] <= 0 : retourne 0
    return np.maximum(0, Z)


def relu_derivative(Z):
    """
    Derivee de ReLU :
        1 si z > 0
        0 si z <= 0
    Utilisee pendant la backpropagation.

    Parametre :
        Z : np.ndarray (valeurs AVANT activation)

    Retourne :
        matrice de 0 et de 1, meme shape que Z
    """
    # (Z > 0) cree une matrice booleenne True/False
    # .astype(float) convertit True -> 1.0 et False -> 0.0
    return (Z > 0).astype(float)


def softmax(Z):
    """
    Fonction softmax : convertit des scores en probabilites.
    Utilisee en couche de sortie pour classification multi-classes.
    La somme des sorties pour chaque exemple = 1.

    Parametre :
        Z : np.ndarray de shape (n_classes, m)
            n_classes = nombre de classes
            m = nombre d'exemples

    Retourne :
        meme shape que Z, chaque colonne somme a 1
    """
    # On soustrait le max de chaque colonne pour la stabilite numerique
    # (evite que e^(grand nombre) devienne infini)
    # keepdims=True garde la shape (n_classes, 1) pour le broadcast
    Z_stable = Z - np.max(Z, axis=0, keepdims=True)

    # Calcul de e^z pour chaque element
    exp_Z = np.exp(Z_stable)

    # Division par la somme des exponentielles de chaque colonne
    # np.sum(..., axis=0, keepdims=True) : somme sur les lignes (classes)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [ ]:
#Initialisation des parametres
def initialize_parameters(layer_dims):
    """
    Initialise les poids W et biais b pour toutes les couches.
    Utilise l'initialisation Xavier/He pour eviter :
      - le vanishing gradient (gradients qui disparaissent)
      - l'exploding gradient (gradients qui explosent)

    Parametre :
        layer_dims : liste des tailles de chaque couche
            Exemple : [4, 5, 3, 1]
            => couche entree = 4 neurones
            => couche cachee 1 = 5 neurones
            => couche cachee 2 = 3 neurones
            => couche sortie = 1 neurone

    Retourne :
        params : dictionnaire contenant W1, b1, W2, b2, ..., WL, bL
    """
    params = {}              # dictionnaire vide pour stocker les parametres

    # len(layer_dims) donne le nombre total de couches (entree incluse)
    L = len(layer_dims)

    # On parcourt les couches de 1 a L-1 (pas la couche d'entree)
    for l in range(1, L):

        # --- Poids W[l] ---
        # Shape : (nombre de neurones de la couche l,
        #          nombre de neurones de la couche l-1)
        # np.random.randn(...) : valeurs aleatoires selon loi normale (moyenne=0, std=1)
        # * np.sqrt(2 / layer_dims[l-1]) : facteur Xavier/He
        #   -> reduit la variance pour eviter explosion/vanissement du gradient
        params['W' + str(l)] = np.random.randn(
            layer_dims[l],       # lignes = nb neurones de la couche l
            layer_dims[l - 1]    # colonnes = nb neurones de la couche l-1
        ) * np.sqrt(2 / layer_dims[l - 1])

        # --- Biais b[l] ---
        # Shape : (nombre de neurones de la couche l, 1)
        # np.zeros : initialise a zero (standard pour les biais)
        # keepdims implicite : shape (n_h, 1) pas (n_h,)
        params['b' + str(l)] = np.zeros((layer_dims[l], 1))

    return params


In [ ]:
#Tests unitaires
print("=" * 50)
print("TEST 1 : Fonctions d'activation")
print("=" * 50)

# Test sigmoid
z_test = np.array([-2.0, 0.0, 2.0])
print(f"\nsigmoid([-2, 0, 2]) = {sigmoid(z_test)}")
print(f"Attendu environ    : [0.119, 0.500, 0.881]")

# Test sigmoid_derivative
print(f"\nsigmoid_derivative([0]) = {sigmoid_derivative(np.array([0.0]))}")
print(f"Attendu            : [0.25]  (valeur max de la derivee)")


In [ ]:
# Test relu
z_test2 = np.array([-3.0, -1.0, 0.0, 2.0, 5.0])
print(f"\nrelu([-3,-1,0,2,5]) = {relu(z_test2)}")
print(f"Attendu             : [0, 0, 0, 2, 5]")

# Test relu_derivative
print(f"\nrelu_derivative([-1, 0, 3]) = {relu_derivative(np.array([-1.0, 0.0, 3.0]))}")
print(f"Attendu                     : [0, 0, 1]")


In [ ]:
# Test softmax
z_soft = np.array([[1.0], [2.0], [0.5]])  # shape (3, 1) : 3 classes, 1 exemple
probs = softmax(z_soft)
print(f"\nsoftmax([[1.0],[2.0],[0.5]]) =\n{probs}")
print(f"Somme des probabilites      = {np.sum(probs):.4f}  (doit valoir 1.0)")


In [ ]:
print("\n" + "=" * 50)
print("TEST 2 : Initialisation des parametres")
print("=" * 50)

# Exemple : reseau [3, 4, 2]
# entree = 3 features, 1 couche cachee de 4 neurones, sortie = 2 classes
layer_dims_test = [3, 4, 2]
params_test = initialize_parameters(layer_dims_test)

print(f"\nArchitecture testee : {layer_dims_test}")
print(f"\nW1 shape : {params_test['W1'].shape}  (attendu : (4, 3))")
print(f"b1 shape : {params_test['b1'].shape}  (attendu : (4, 1))")
print(f"W2 shape : {params_test['W2'].shape}  (attendu : (2, 4))")
print(f"b2 shape : {params_test['b2'].shape}  (attendu : (2, 1))")

print(f"\nW1 (premiers poids initialises) :\n{params_test['W1']}")
print(f"\nW2 :\n{params_test['W2']}")


In [ ]:

print("\n" + "=" * 50)
print("TEST 3 : Pourquoi PAS initialiser a zero les poids ?")
print("=" * 50)

# Si on initialise tous les W a zero :
params_zero = {}
params_zero['W1'] = np.zeros((4, 3))   # tous a zero
params_zero['b1'] = np.zeros((4, 1))
# Tous les neurones cachet auront la meme sortie -> ils apprendront la meme chose
# Le reseau se comporte comme s'il n'avait qu'un seul neurone par couche
# C'est le "probleme de symetrie"
print("\nSi W1 = zeros :")
X_test = np.array([[0.5], [0.8], [0.3]])  # 1 exemple avec 3 features
Z1_zero = params_zero['W1'] @ X_test + params_zero['b1']
print(f"Z1 = {Z1_zero.T}  <- tous identiques (symetrie brisee impossible)")
print("-> C'est pourquoi on initialise avec des valeurs aleatoires petites !")

print("\nSi W1 = Xavier aleatoire :")
Z1_xavier = params_test['W1'] @ X_test + params_test['b1']
print(f"Z1 = {Z1_xavier.T}  <- tous differents (chaque neurone apprend differemment)")


#Forward Pass (Propagation avant)

In [ ]:
#Forward Pass complet (propagation avant)


def linear_forward(A_prev, W, b):
    """
    Calcule la pre-activation Z d'UNE seule couche.
    C'est la partie lineaire : Z = W @ A_prev + b

    Parametres :
        A_prev : activation de la couche precedente
                 shape (n_{l-1}, m)
                 n_{l-1} = nb neurones couche precedente
                 m = nb d'exemples
        W      : matrice des poids de la couche actuelle
                 shape (n_l, n_{l-1})
        b      : vecteur des biais de la couche actuelle
                 shape (n_l, 1)

    Retourne :
        Z      : pre-activation, shape (n_l, m)
        cache  : tuple (A_prev, W, b) stocke pour la backprop
    """
    # Produit matriciel : W de shape (n_l, n_{l-1})
    #                  @ A_prev de shape (n_{l-1}, m)
    #                  = Z de shape (n_l, m)
    # NumPy broadcaste b de shape (n_l, 1) sur toutes les m colonnes
    Z = W @ A_prev + b

    # On stocke A_prev, W, b dans le cache
    # La backprop en aura besoin pour calculer les gradients
    cache = (A_prev, W, b)

    return Z, cache


def activation_forward(A_prev, W, b, activation):
    """
    Calcule Z = W @ A_prev + b, puis applique la fonction d'activation.
    Combine la partie lineaire ET l'activation en une seule fonction.

    Parametres :
        A_prev     : activation de la couche precedente, shape (n_{l-1}, m)
        W          : poids, shape (n_l, n_{l-1})
        b          : biais, shape (n_l, 1)
        activation : chaine de caracteres, 'relu', 'sigmoid' ou 'softmax'

    Retourne :
        A     : activation apres la fonction choisie, shape (n_l, m)
        cache : dictionnaire avec 'linear' et 'Z' pour la backprop
    """
    # Etape 1 : calcul lineaire Z = W @ A_prev + b
    Z, linear_cache = linear_forward(A_prev, W, b)

    # Etape 2 : application de la fonction d'activation choisie
    if activation == 'relu':
        # ReLU pour les couches cachees
        A = relu(Z)

    elif activation == 'sigmoid':
        # Sigmoide pour la sortie en classification binaire
        A = sigmoid(Z)

    elif activation == 'softmax':
        # Softmax pour la sortie en classification multi-classes
        A = softmax(Z)

    else:
        # Si le nom de l'activation est inconnu -> erreur claire
        raise ValueError(f"Activation '{activation}' inconnue. "
                         f"Choisir 'relu', 'sigmoid' ou 'softmax'.")

    # On stocke dans le cache :
    # - linear_cache = (A_prev, W, b) pour calculer dW et db
    # - Z pour calculer la derivee de l'activation (dZ)
    cache = {
        'linear': linear_cache,   # (A_prev, W, b)
        'Z': Z                    # pre-activation, utile pour ReLU'(Z)
    }

    return A, cache


def forward_propagation(X, params, output_activation='softmax'):
    """
    Effectue le forward pass complet a travers toutes les couches.
    Couches cachees : ReLU
    Couche de sortie : softmax (multi-classes) ou sigmoid (binaire)

    Parametres :
        X                 : donnees d'entree, shape (n_x, m)
        params            : dictionnaire des poids W et biais b
                            cles : 'W1','b1','W2','b2',...,'WL','bL'
        output_activation : 'softmax' ou 'sigmoid' pour la derniere couche

    Retourne :
        A_final : predictions du reseau, shape (n_y, m)
        caches  : liste de tous les caches de chaque couche
                  [cache_couche_1, cache_couche_2, ..., cache_couche_L]
                  Chaque cache contient 'linear' et 'Z'
    """
    caches = []   # liste pour stocker les caches de chaque couche

    # L'activation initiale est les donnees d'entree elles-memes
    # (la "couche 0" ne fait rien d'autre que transmettre X)
    A = X

    # Nombre de couches = nombre de paires (W, b) dans params
    # params contient W1,b1,W2,b2,...,WL,bL
    # len(params) // 2 donne le nombre de couches
    L = len(params) // 2

    # --- Couches cachees (de la couche 1 a L-1) avec ReLU ---
    for l in range(1, L):
        A_prev = A   # sauvegarde de l'activation precedente

        # Recupere les poids et biais de la couche l
        # str(l) convertit l'entier en chaine pour acceder au dict
        W = params['W' + str(l)]
        b = params['b' + str(l)]

        # Forward pour la couche l avec activation ReLU
        A, cache = activation_forward(A_prev, W, b, activation='relu')

        # Stocke le cache de cette couche dans la liste
        caches.append(cache)

    # --- Couche de sortie (couche L) avec softmax ou sigmoid ---
    W_final = params['W' + str(L)]   # poids de la derniere couche
    b_final = params['b' + str(L)]   # biais de la derniere couche

    A_final, cache_final = activation_forward(
        A, W_final, b_final, activation=output_activation
    )

    # Stocke aussi le cache de la couche de sortie
    caches.append(cache_final)

    # Verification de la shape de sortie (pour debugger facilement)
    # A_final doit etre (n_y, m)
    assert A_final.shape == (W_final.shape[0], X.shape[1]), \
        f"Erreur de shape : A_final est {A_final.shape}, " \
        f"attendu ({W_final.shape[0]}, {X.shape[1]})"

    return A_final, caches

Fonction de cout (Loss)

In [ ]:
#Fonction de cout (Loss)
def compute_loss(A_final, Y, loss_type='cross_entropy_multi'):
    """
    Calcule la fonction de cout (loss) entre les predictions et les vraies valeurs.

    Parametres :
        A_final   : predictions du reseau, shape (n_y, m)
        Y         : vraies valeurs one-hot, shape (n_y, m)
        loss_type : 'cross_entropy_multi' (defaut) ou 'cross_entropy_binary'

    Retourne :
        loss : scalaire (valeur de la loss pour cet iteration)
    """
    # m = nombre d'exemples
    m = Y.shape[1]

    if loss_type == 'cross_entropy_multi':
        # Cross-entropy multi-classes :
        # L = -(1/m) * sum(Y * log(A_final))
        # np.clip evite log(0) qui vaut -infini
        A_clip = np.clip(A_final, 1e-8, 1 - 1e-8)

        # Y * np.log(A_clip) : produit element par element
        # np.sum(...) : somme sur toutes les classes ET tous les exemples
        loss = -np.sum(Y * np.log(A_clip)) / m

    elif loss_type == 'cross_entropy_binary':
        # Cross-entropy binaire (pour sortie sigmoide, 1 neurone de sortie) :
        # L = -(1/m) * sum(Y*log(A) + (1-Y)*log(1-A))
        A_clip = np.clip(A_final, 1e-8, 1 - 1e-8)

        loss = -np.sum(
            Y * np.log(A_clip) + (1 - Y) * np.log(1 - A_clip)
        ) / m

    else:
        raise ValueError(f"loss_type '{loss_type}' inconnu.")

    # np.squeeze supprime les dimensions inutiles (ex: shape (1,1) -> scalaire)
    return float(np.squeeze(loss))



 Tests complets du Forward Pass

In [ ]:
 #Tests complets du Forward Pass
print("=" * 55)
print("TEST 1 : Forward pass sur un mini-exemple a la main")
print("=" * 55)

# Reseau : 2 entrees -> 2 neurones caches -> 1 sortie (sigmoid)
# Memes valeurs que l'exemple du Jour 1 et Jour 3 pour verifier

# Donnees : 1 exemple, 2 features
X_mini = np.array([[0.5],
                   [0.8]])   # shape (2, 1)

# Parametres fixes a la main (pas aleatoires) pour verifier
params_mini = {
    'W1': np.array([[0.3, -0.2],
                    [0.5,  0.7]]),   # shape (2, 2)
    'b1': np.array([[0.0],
                    [0.0]]),          # shape (2, 1)
    'W2': np.array([[0.6, -0.4]]),   # shape (1, 2)
    'b2': np.array([[0.0]])          # shape (1, 1)
}

# Forward pass avec sigmoid en sortie (classification binaire)
A_mini, caches_mini = forward_propagation(
    X_mini, params_mini, output_activation='sigmoid'
)

print(f"\nEntree X                   : {X_mini.T}")
print(f"\nZ1 (cache couche 1)        : {caches_mini[0]['Z'].T}")
print(f"A1 = ReLU(Z1)              : {caches_mini[0]['linear'][0].T}")
# Note : A1 est stocke comme A_prev dans le cache de la couche 2
print(f"A1 via couche 2 cache      : {caches_mini[1]['linear'][0].T}")
print(f"\nZ2 (cache couche 2)        : {caches_mini[1]['Z'].T}")
print(f"A2 = sigmoid(Z2) = y_hat   : {A_mini.T}")
print(f"\nAttendu (calcul Jour 3)    : A2 ~= 0.433")


print("\n" + "=" * 55)
print("TEST 2 : Forward pass multi-classes sur Iris")
print("=" * 55)

# Chargement du dataset Iris (sans sklearn pour l'instant)
# On simule 4 exemples avec 4 features (Iris a 4 features)
np.random.seed(0)
X_iris_sample = np.random.randn(4, 10)   # 4 features, 10 exemples
# Labels one-hot pour 3 classes (Iris a 3 classes)
Y_iris_sample = np.zeros((3, 10))
for i in range(10):
    Y_iris_sample[np.random.randint(0, 3), i] = 1

# Architecture : [4, 8, 3]
# entree=4 features, 1 couche cachee=8 neurones, sortie=3 classes
layer_dims_iris = [4, 8, 3]
params_iris = initialize_parameters(layer_dims_iris)

# Forward pass
A_iris, caches_iris = forward_propagation(
    X_iris_sample, params_iris, output_activation='softmax'
)

print(f"\nArchitecture               : {layer_dims_iris}")
print(f"X shape                    : {X_iris_sample.shape}  (4 features, 10 exemples)")
print(f"A_final shape              : {A_iris.shape}  (3 classes, 10 exemples)")
print(f"\nPredictions (10 exemples) :")
print(np.round(A_iris, 3))
print(f"\nSomme par colonne (doit = 1.0) :")
print(np.round(np.sum(A_iris, axis=0), 4))


print("\n" + "=" * 55)
print("TEST 3 : Calcul de la loss")
print("=" * 55)

loss_val = compute_loss(A_iris, Y_iris_sample, loss_type='cross_entropy_multi')
print(f"\nLoss initiale (avant entrainement) : {loss_val:.4f}")
print(f"Attendu : proche de log(3) = {np.log(3):.4f}")
print("(Un reseau aleatoire sur 3 classes predit ~1/3 chaque classe)")
print("=> La loss doit diminuer pendant l'entrainement")


print("\n" + "=" * 55)
print("TEST 4 : Verification des shapes a chaque couche")
print("=" * 55)

print(f"\nArchitecture testee : {layer_dims_iris}")
print(f"Nombre d'exemples m = {X_iris_sample.shape[1]}")
print()

# Affiche les shapes de chaque couche
for l, cache in enumerate(caches_iris):
    A_prev_shape = cache['linear'][0].shape   # A_prev
    W_shape      = cache['linear'][1].shape   # W
    b_shape      = cache['linear'][2].shape   # b
    Z_shape      = cache['Z'].shape           # Z

    print(f"Couche {l+1} :")
    print(f"  A_prev shape : {A_prev_shape}")
    print(f"  W shape      : {W_shape}")
    print(f"  b shape      : {b_shape}")
    print(f"  Z shape      : {Z_shape}")
    print()

Backward Pass (Retropropagation)

In [ ]:
#Backward Pass : couche par couche
def linear_backward(dZ, cache):
    """
    Calcule les gradients de la partie LINEAIRE d'une couche.
    Rappel : Z = W @ A_prev + b
    On cherche dW, db, et dA_prev a partir de dZ.

    Parametres :
        dZ    : gradient de la loss par rapport a Z
                shape (n_l, m)
        cache : tuple (A_prev, W, b) stocke pendant le forward pass

    Retourne :
        dA_prev : gradient par rapport a A_prev, shape (n_{l-1}, m)
                  permet de continuer a remonter vers la couche precedente
        dW      : gradient par rapport a W, shape (n_l, n_{l-1})
                  meme shape que W -> utilisé pour mettre a jour W
        db      : gradient par rapport a b, shape (n_l, 1)
                  meme shape que b -> utilisé pour mettre a jour b
    """
    # On recupere ce qui avait ete stocke pendant le forward
    A_prev, W, b = cache

    # m = nombre d'exemples (taille de la dimension 1)
    m = A_prev.shape[1]

    # --- dW = (1/m) * dZ @ A_prev.T ---
    # dZ shape      : (n_l, m)
    # A_prev.T shape: (m, n_{l-1})
    # dZ @ A_prev.T : (n_l, n_{l-1}) = meme shape que W -> correct
    # Division par m pour faire la moyenne sur tous les exemples
    dW = (1 / m) * dZ @ A_prev.T

    # --- db = (1/m) * sum(dZ, axis=1) ---
    # On somme dZ sur tous les exemples (axis=1 = dimension des colonnes)
    # keepdims=True garde la shape (n_l, 1) au lieu de (n_l,)
    # Sans keepdims, db aurait shape (n_l,) et causerait des bugs de broadcast
    db = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

    # --- dA_prev = W.T @ dZ ---
    # W.T shape : (n_{l-1}, n_l)
    # dZ shape  : (n_l, m)
    # W.T @ dZ  : (n_{l-1}, m) = meme shape que A_prev -> correct
    # Ce gradient sera passe a la couche precedente pour continuer la backprop
    dA_prev = W.T @ dZ

    return dA_prev, dW, db


def activation_backward(dA, cache, activation):
    """
    Calcule les gradients a travers la fonction d'activation.
    Applique la chain rule : dZ = dA * g'(Z)
    Puis appelle linear_backward pour obtenir dW, db, dA_prev.

    Parametres :
        dA         : gradient de la loss par rapport a A (apres activation)
                     shape (n_l, m)
        cache      : dictionnaire {'linear': (A_prev, W, b), 'Z': Z}
        activation : 'relu', 'sigmoid' ou 'softmax_cross_entropy'

    Retourne :
        dA_prev : gradient vers la couche precedente, shape (n_{l-1}, m)
        dW      : gradient pour W, shape (n_l, n_{l-1})
        db      : gradient pour b, shape (n_l, 1)
    """
    # On recupere le cache stocke pendant le forward pass
    linear_cache = cache['linear']   # (A_prev, W, b)
    Z = cache['Z']                   # pre-activation

    if activation == 'relu':
        # Chain rule : dZ = dA * ReLU'(Z)
        # ReLU'(Z) = 1 si Z > 0, 0 sinon
        # Multiplication element par element (pas de produit matriciel ici)
        dZ = dA * relu_derivative(Z)

    elif activation == 'sigmoid':
        # Chain rule : dZ = dA * sigma'(Z)
        # sigma'(Z) = sigma(Z) * (1 - sigma(Z))
        dZ = dA * sigmoid_derivative(Z)

    elif activation == 'softmax_cross_entropy':
        # Cas special : softmax + cross-entropy multi-classes
        # Le gradient combine donne simplement dZ = A - Y
        # dA contient deja (A - Y) calcule dans backward_propagation
        # On le passe directement sans multiplier par une derivee
        dZ = dA

    else:
        raise ValueError(f"Activation '{activation}' inconnue pour la backprop.")

    # Maintenant qu'on a dZ, on calcule dW, db, dA_prev
    dA_prev, dW, db = linear_backward(dZ, linear_cache)

    return dA_prev, dW, db


def backward_propagation(A_final, Y, caches, output_activation='softmax'):
    """
    Effectue le backward pass complet : calcule les gradients
    de la loss par rapport a tous les poids W et biais b.

    Parametres :
        A_final           : predictions du reseau, shape (n_y, m)
        Y                 : vraies valeurs one-hot, shape (n_y, m)
        caches            : liste des caches du forward pass
                            [cache_couche_1, ..., cache_couche_L]
        output_activation : 'softmax' ou 'sigmoid'

    Retourne :
        grads : dictionnaire avec dW1, db1, dW2, db2, ..., dWL, dbL
    """
    grads = {}         # dictionnaire pour stocker tous les gradients
    L = len(caches)    # nombre total de couches
    m = Y.shape[1]     # nombre d'exemples

    # -------------------------------------------------------
    # ETAPE 1 : Gradient de la loss par rapport a A_final
    # -------------------------------------------------------

    if output_activation == 'softmax':
        # Pour softmax + cross-entropy multi-classes :
        # Le gradient combine est simplement : dZ_final = A_final - Y
        # On le stocke directement dans dA_final pour qu'il soit
        # passe tel quel a activation_backward (cas 'softmax_cross_entropy')
        dA_final = A_final - Y

    elif output_activation == 'sigmoid':
        # Pour sigmoid + cross-entropy binaire :
        # dL/dA = -Y/A + (1-Y)/(1-A)
        A_clip = np.clip(A_final, 1e-8, 1 - 1e-8)
        dA_final = -(Y / A_clip) + (1 - Y) / (1 - A_clip)

    # -------------------------------------------------------
    # ETAPE 2 : Backward pass de la couche de sortie (couche L)
    # -------------------------------------------------------

    # Le cache de la couche de sortie est le dernier de la liste
    cache_final = caches[L - 1]

    # Choix de l'activation pour la backprop de la derniere couche
    if output_activation == 'softmax':
        activation_out = 'softmax_cross_entropy'
    else:
        activation_out = 'sigmoid'

    # Calcul des gradients de la couche de sortie
    dA_prev, dW_final, db_final = activation_backward(
        dA_final, cache_final, activation=activation_out
    )

    # Stockage dans grads avec la cle 'WL' et 'bL'
    grads['dW' + str(L)] = dW_final
    grads['db' + str(L)] = db_final

    # -------------------------------------------------------
    # ETAPE 3 : Backward pass des couches cachees (L-1 a 1)
    # -------------------------------------------------------

    # On remonte de la couche L-1 jusqu'a la couche 1
    # reversed(range(L-1)) donne : L-2, L-3, ..., 1, 0
    for l in reversed(range(L - 1)):

        # Cache de la couche l+1 (indices Python commencent a 0)
        cache_l = caches[l]

        # dA_prev contient le gradient remontant de la couche suivante
        # On l'utilise pour calculer les gradients de cette couche
        dA_prev, dW_l, db_l = activation_backward(
            dA_prev, cache_l, activation='relu'
        )

        # Stockage : l+1 car la couche 0 de Python = couche 1 du reseau
        grads['dW' + str(l + 1)] = dW_l
        grads['db' + str(l + 1)] = db_l

    return grads



In [ ]:
#Mise a jour des parametres (SGD)
def update_parameters(params, grads, learning_rate):
    """
    Met a jour tous les poids et biais avec la descente de gradient (SGD).
    Formule : W = W - learning_rate * dW
              b = b - learning_rate * db

    Parametres :
        params        : dictionnaire des parametres actuels (W1,b1,...,WL,bL)
        grads         : dictionnaire des gradients (dW1,db1,...,dWL,dbL)
        learning_rate : taux d'apprentissage alpha (ex: 0.01)

    Retourne :
        params : dictionnaire mis a jour (nouveaux W et b)
    """
    # Nombre de couches
    L = len(params) // 2

    for l in range(1, L + 1):
        # W[l] = W[l] - alpha * dW[l]
        # params['W1'] -= learning_rate * grads['dW1'] (in-place)
        params['W' + str(l)] = (
            params['W' + str(l)] - learning_rate * grads['dW' + str(l)]
        )

        # b[l] = b[l] - alpha * db[l]
        params['b' + str(l)] = (
            params['b' + str(l)] - learning_rate * grads['db' + str(l)]
        )

    return params


In [ ]:
#Tests complets du Backward Pass
print("=" * 55)
print("TEST 1 : Backward pass sur mini-exemple (Jour 3)")
print("=" * 55)

# Meme mini-exemple que le Jour 3 pour verifier les gradients
X_mini = np.array([[0.5], [0.8]])
Y_mini = np.array([[1.0]])   # Y=1, classification binaire

params_mini = {
    'W1': np.array([[0.3, -0.2], [0.5, 0.7]]),
    'b1': np.array([[0.0], [0.0]]),
    'W2': np.array([[0.6, -0.4]]),
    'b2': np.array([[0.0]])
}

# Forward
A_mini, caches_mini = forward_propagation(
    X_mini, params_mini, output_activation='sigmoid'
)
print(f"\nA_final (y_hat) = {A_mini[0,0]:.4f}  (attendu ~0.433 du Jour 3)")

# Loss
loss_mini = compute_loss(A_mini, Y_mini, loss_type='cross_entropy_binary')
print(f"Loss            = {loss_mini:.4f}  (attendu ~0.836)")

# Backward
grads_mini = backward_propagation(
    A_mini, Y_mini, caches_mini, output_activation='sigmoid'
)

print(f"\ndZ2 = A2 - Y   = {A_mini[0,0] - Y_mini[0,0]:.4f}  (attendu -0.567)")
print(f"\ndW2            = {grads_mini['dW2']}")
print(f"Attendu Jour 3 : [[-0.051  -0.459]]")
print(f"\ndW1            =\n{grads_mini['dW1']}")
print(f"Attendu Jour 3 :\n[[-0.170  -0.272]\n [ 0.114   0.182]]")


print("\n" + "=" * 55)
print("TEST 2 : Verification des shapes des gradients")
print("=" * 55)

# Architecture [4, 8, 3] sur Iris simule
np.random.seed(0)
X_iris = np.random.randn(4, 30)
Y_iris = np.zeros((3, 30))
for i in range(30):
    Y_iris[np.random.randint(0, 3), i] = 1

layer_dims = [4, 8, 3]
params = initialize_parameters(layer_dims)

A_out, caches = forward_propagation(X_iris, params, output_activation='softmax')
grads = backward_propagation(A_out, Y_iris, caches, output_activation='softmax')

print(f"\nArchitecture : {layer_dims}")
print()
for key in sorted(grads.keys()):
    param_key = key.replace('d', '')   # 'dW1' -> 'W1'
    g_shape = grads[key].shape
    p_shape = params[param_key].shape
    match = "OK" if g_shape == p_shape else "ERREUR"
    print(f"  {key} shape : {g_shape}  | {param_key} shape : {p_shape}  [{match}]")


print("\n" + "=" * 55)
print("TEST 3 : Mise a jour et verification que la loss diminue")
print("=" * 55)

# On verifie qu'apres une mise a jour, la loss diminue bien
loss_avant = compute_loss(A_out, Y_iris, loss_type='cross_entropy_multi')

# Une mise a jour SGD
params = update_parameters(params, grads, learning_rate=0.1)

# Nouveau forward pass avec les poids mis a jour
A_out_new, _ = forward_propagation(X_iris, params, output_activation='softmax')
loss_apres = compute_loss(A_out_new, Y_iris, loss_type='cross_entropy_multi')

print(f"\nLoss AVANT mise a jour : {loss_avant:.6f}")
print(f"Loss APRES mise a jour : {loss_apres:.6f}")

if loss_apres < loss_avant:
    print("=> La loss a diminue. La backprop fonctionne correctement !")
else:
    print("=> ATTENTION : la loss n'a pas diminue. Verifier le code.")


print("\n" + "=" * 55)
print("TEST 4 : Gradient check (verification numerique)")
print("=" * 55)

# Le gradient check est une technique pour verifier que la backprop
# est correcte : on compare le gradient calcule par backprop
# avec une approximation numerique (definition de la derivee)
# Si les deux sont proches, la backprop est correcte.

# On reinitialise pour un test propre
np.random.seed(1)
X_check = np.random.randn(3, 5)
Y_check = np.zeros((2, 5))
for i in range(5):
    Y_check[np.random.randint(0, 2), i] = 1

params_check = initialize_parameters([3, 4, 2])

A_check, caches_check = forward_propagation(
    X_check, params_check, output_activation='softmax'
)
grads_check = backward_propagation(
    A_check, Y_check, caches_check, output_activation='softmax'
)

# Approximation numerique du gradient de W1[0,0]
epsilon = 1e-5   # petit pas pour l'approximation

# On perturbe W1[0,0] de +epsilon
params_plus = {k: v.copy() for k, v in params_check.items()}
params_plus['W1'][0, 0] += epsilon
A_plus, _ = forward_propagation(X_check, params_plus, output_activation='softmax')
loss_plus = compute_loss(A_plus, Y_check)

# On perturbe W1[0,0] de -epsilon
params_minus = {k: v.copy() for k, v in params_check.items()}
params_minus['W1'][0, 0] -= epsilon
A_minus, _ = forward_propagation(X_check, params_minus, output_activation='softmax')
loss_minus = compute_loss(A_minus, Y_check)

# Approximation numerique : (f(x+e) - f(x-e)) / (2*e)
grad_numerique = (loss_plus - loss_minus) / (2 * epsilon)

# Gradient calcule par backprop
grad_backprop = grads_check['dW1'][0, 0]

# Difference relative
diff = abs(grad_numerique - grad_backprop) / (
    abs(grad_numerique) + abs(grad_backprop) + 1e-8
)

print(f"\nGradient numerique  dW1[0,0] : {grad_numerique:.8f}")
print(f"Gradient backprop   dW1[0,0] : {grad_backprop:.8f}")
print(f"Difference relative          : {diff:.2e}")

if diff < 1e-5:
    print("=> Gradient check OK : backprop correcte !")
else:
    print("=> ATTENTION : difference trop grande, verifier la backprop.")


JOUR 7 — Entrainement complet sur le dataset Iris

In [ ]:
#Preparation du dataset Iris
print("=" * 55)
print("ETAPE 1 : Chargement et preparation du dataset Iris")
print("=" * 55)

# Chargement du dataset Iris
# Iris contient 150 exemples, 4 features, 3 classes
iris = load_iris()

# X : matrice des features, shape (150, 4)
# y : vecteur des labels, shape (150,)  valeurs : 0, 1 ou 2
X_raw = iris.data    # 4 features : longueur/largeur sepale et petale
y_raw = iris.target  # 3 classes : setosa=0, versicolor=1, virginica=2

print(f"\nDataset Iris :")
print(f"  X shape : {X_raw.shape}  (150 exemples, 4 features)")
print(f"  y shape : {y_raw.shape}  (150 labels)")
print(f"  Classes : {iris.target_names}")
print(f"  Features : {iris.feature_names}")

# --- Split train/test : 80% entrainement, 20% test ---
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw,
    test_size=0.2,       # 20% pour le test = 30 exemples
    random_state=42,     # graine pour reproductibilite
    stratify=y_raw       # garder les proportions de classes dans chaque split
)

print(f"\nApres split :")
print(f"  X_train shape : {X_train_raw.shape}  (120 exemples)")
print(f"  X_test shape  : {X_test_raw.shape}   (30 exemples)")

# --- Normalisation avec StandardScaler ---
# Pourquoi normaliser ?
# Sans normalisation, les features avec de grandes valeurs
# dominent le calcul -> le reseau apprend mal
# StandardScaler : centre (moyenne=0) et reduit (std=1)
scaler = StandardScaler()

# fit_transform sur train : calcule moyenne et std sur train UNIQUEMENT
# puis transforme train
X_train_scaled = scaler.fit_transform(X_train_raw)

# transform sur test : utilise la moyenne et std du TRAIN
# (ne jamais fitter le scaler sur le test -> fuite de donnees)
X_test_scaled = scaler.transform(X_test_raw)

print(f"\nApres normalisation (StandardScaler) :")
print(f"  X_train moyenne : {X_train_scaled.mean(axis=0).round(4)}")
print(f"  X_train std     : {X_train_scaled.std(axis=0).round(4)}")

# --- Transposition pour NumPy ---
# Notre reseau attend shape (n_features, m) pas (m, n_features)
# On transpose : (120, 4) -> (4, 120)
X_train = X_train_scaled.T   # shape (4, 120)
X_test  = X_test_scaled.T    # shape (4, 30)

print(f"\nApres transposition (format reseau) :")
print(f"  X_train shape : {X_train.shape}  (4 features, 120 exemples)")
print(f"  X_test shape  : {X_test.shape}   (4 features, 30 exemples)")

# --- One-Hot Encoding des labels ---
# Notre reseau a 3 neurones en sortie (1 par classe)
# y_train contient [0, 2, 1, ...] -> on convertit en :
# 0 -> [1, 0, 0]
# 1 -> [0, 1, 0]
# 2 -> [0, 0, 1]

def one_hot_encode(y, n_classes):
    """
    Convertit un vecteur de labels en matrice one-hot.

    Parametres :
        y         : vecteur de labels entiers, shape (m,)
        n_classes : nombre de classes

    Retourne :
        Y_onehot : matrice one-hot, shape (n_classes, m)
    """
    m = len(y)
    # Matrice de zeros de shape (n_classes, m)
    Y_onehot = np.zeros((n_classes, m))

    # Pour chaque exemple i, on met 1 a la ligne correspondant a sa classe
    # np.arange(m) = [0, 1, 2, ..., m-1] : indices des colonnes
    # y = [0, 2, 1, ...] : indices des lignes
    Y_onehot[y, np.arange(m)] = 1

    return Y_onehot

Y_train = one_hot_encode(y_train, n_classes=3)  # shape (3, 120)
Y_test  = one_hot_encode(y_test,  n_classes=3)  # shape (3, 30)

print(f"\nApres One-Hot Encoding :")
print(f"  Y_train shape : {Y_train.shape}  (3 classes, 120 exemples)")
print(f"  Y_test shape  : {Y_test.shape}   (3 classes, 30 exemples)")
print(f"\n  Exemple : y_train[0]={y_train[0]} -> Y_train[:,0]={Y_train[:,0]}")



In [ ]:
#Fonction de prediction et d'accuracy
def predict(X, params, output_activation='softmax'):
    """
    Fait une prediction pour chaque exemple de X.

    Parametres :
        X                 : donnees, shape (n_x, m)
        params            : parametres du reseau
        output_activation : 'softmax' ou 'sigmoid'

    Retourne :
        predictions : vecteur des classes predites, shape (m,)
                      valeurs : 0, 1, 2, ...
    """
    # Forward pass pour obtenir les probabilites
    A_final, _ = forward_propagation(X, params, output_activation)

    # np.argmax retourne l'indice de la valeur maximale sur axis=0
    # Pour softmax : la classe predite = celle avec la plus grande proba
    # A_final shape : (n_classes, m)
    # np.argmax(..., axis=0) : shape (m,) -> 1 prediction par exemple
    predictions = np.argmax(A_final, axis=0)

    return predictions


def compute_accuracy(X, Y_onehot, params, output_activation='softmax'):
    """
    Calcule l'accuracy (taux de bonnes predictions).

    Parametres :
        X           : donnees, shape (n_x, m)
        Y_onehot    : vraies valeurs one-hot, shape (n_classes, m)
        params      : parametres du reseau
        output_activation : 'softmax' ou 'sigmoid'

    Retourne :
        accuracy : flottant entre 0 et 1
    """
    # Predictions du reseau : shape (m,)
    predictions = predict(X, params, output_activation)

    # Vraies classes : convertir one-hot -> entiers
    # np.argmax sur axis=0 : indice de la ligne qui vaut 1 pour chaque exemple
    true_labels = np.argmax(Y_onehot, axis=0)   # shape (m,)

    # Comparaison : (predictions == true_labels) -> tableau de True/False
    # np.mean(...) : proportion de True = accuracy
    accuracy = np.mean(predictions == true_labels)

    return accuracy


In [ ]:
# Boucle d'entrainement complete
def model(X_train, Y_train, X_test, Y_test,
          layer_dims, learning_rate=0.01,
          num_iterations=2000, print_every=200,
          output_activation='softmax', verbose=True):  # <-- verbose ajoute ici
    """
    Entraine le reseau de neurones from scratch.
    Assemble : initialisation -> boucle (forward, loss, backward, update)

    Parametres :
        X_train, Y_train  : donnees d'entrainement
        X_test, Y_test    : donnees de test (pour suivi)
        layer_dims        : architecture [n_x, n_h1, ..., n_y]
        learning_rate     : taux d'apprentissage alpha
        num_iterations    : nombre d'iterations (epochs)
        print_every       : affiche la loss tous les N iterations
        output_activation : 'softmax' ou 'sigmoid'
        verbose           : True = affiche la progression,
                            False = silencieux (utile pour les tests
                            d'hyperparametres au Jour 10)

    Retourne :
        params       : parametres entraines (W et b finaux)
        loss_history : liste des valeurs de loss a chaque iteration
        acc_history  : liste des accuracies train a chaque iteration
    """
    # --- Initialisation des parametres ---
    params = initialize_parameters(layer_dims)

    # Listes pour stocker l'evolution de la loss et accuracy
    loss_history = []
    acc_history  = []

    # Chronometre
    start_time = time.time()

    # --- Boucle d'entrainement ---
    for i in range(num_iterations):

        # ETAPE 1 : Forward pass
        # Calcule les predictions A_final et stocke les caches
        A_final, caches = forward_propagation(
            X_train, params, output_activation
        )

        # ETAPE 2 : Calcul de la loss
        # Mesure l'erreur entre predictions et vraies valeurs
        if output_activation == 'softmax':
            loss = compute_loss(A_final, Y_train, 'cross_entropy_multi')
        else:
            loss = compute_loss(A_final, Y_train, 'cross_entropy_binary')

        # ETAPE 3 : Backward pass
        # Calcule les gradients de la loss par rapport a W et b
        grads = backward_propagation(
            A_final, Y_train, caches, output_activation
        )

        # ETAPE 4 : Mise a jour des parametres (SGD)
        # Ajuste W et b dans la direction qui reduit la loss
        params = update_parameters(params, grads, learning_rate)

        # --- Sauvegarde de la loss et accuracy ---
        loss_history.append(loss)
        acc = compute_accuracy(X_train, Y_train, params, output_activation)
        acc_history.append(acc)

        # --- Affichage periodique ---
        # verbose=True  : affiche normalement
        # verbose=False : aucun affichage (mode silencieux)
        if verbose and (i % print_every == 0 or i == num_iterations - 1):  # <-- verbose ajoute ici
            acc_test = compute_accuracy(
                X_test, Y_test, params, output_activation
            )
            print(f"Iteration {i:5d} | Loss : {loss:.4f} | "
                  f"Acc train : {acc*100:.1f}% | "
                  f"Acc test : {acc_test*100:.1f}%")

    # Temps total d'entrainement
    elapsed = time.time() - start_time
    if verbose:  # <-- n'affiche le temps que si verbose=True
        print(f"\nTemps d'entrainement : {elapsed:.2f} secondes")

    return params, loss_history, acc_history, elapsed

Lancement de l'entrainement

In [ ]:
#Lancement de l'entrainement
print("\n" + "=" * 55)
print("ETAPE 2 : Entrainement du reseau")
print("=" * 55)

# Architecture : [4, 16, 8, 3]
# Entree = 4 features Iris
# Couche cachee 1 = 16 neurones
# Couche cachee 2 = 8 neurones
# Sortie = 3 classes
layer_dims = [4, 16, 8, 3]

print(f"\nArchitecture : {layer_dims}")
print(f"Learning rate : 0.05")
print(f"Iterations : 2000")
print()

params_trained, loss_history, acc_history, temps = model(
    X_train, Y_train,
    X_test,  Y_test,
    layer_dims        = layer_dims,
    learning_rate     = 0.05,
    num_iterations    = 2000,
    print_every       = 200,
    output_activation = 'softmax'
)


Resultats finaux et visualisations

In [ ]:
print("\n" + "=" * 55)
print("ETAPE 3 : Resultats finaux")
print("=" * 55)

# Accuracy finale sur train et test
acc_train_final = compute_accuracy(
    X_train, Y_train, params_trained, 'softmax'
) * 100

acc_test_final = compute_accuracy(
    X_test, Y_test, params_trained, 'softmax'
) * 100

print(f"\nAccuracy finale sur TRAIN : {acc_train_final:.1f}%")
print(f"Accuracy finale sur TEST  : {acc_test_final:.1f}%")

# --- Predictions detaillees sur le test ---
y_pred = predict(X_test, params_trained, 'softmax')
y_true = np.argmax(Y_test, axis=0)

print(f"\nPredictions vs Vraies valeurs (test) :")
print(f"  Predites : {y_pred}")
print(f"  Vraies   : {y_true}")

# --- Matrice de confusion manuelle ---
print(f"\nMatrice de confusion (test) :")
n_classes = 3
confusion = np.zeros((n_classes, n_classes), dtype=int)
for pred, true in zip(y_pred, y_true):
    confusion[true, pred] += 1

print(f"\n{'':15} Predit 0  Predit 1  Predit 2")
for i in range(n_classes):
    print(f"  Vraie classe {i} :  "
          f"{confusion[i,0]:6d}    {confusion[i,1]:6d}    {confusion[i,2]:6d}")

print(f"\nLecture : ligne = vraie classe, colonne = classe predite")
print(f"Diagonale = bonnes predictions")


Graphiques

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Reseau de neurones from scratch — Dataset Iris\n'
    f'Architecture : {layer_dims} | Learning rate : 0.05',
    fontsize=13, fontweight='bold'
)

# --- Graphique 1 : Courbe de loss ---
axes[0].plot(loss_history, color='#A32D2D', linewidth=2, label='Loss (train)')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Loss (cross-entropy)', fontsize=12)
axes[0].set_title('Evolution de la fonction de cout', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
# Annotation de la loss finale
axes[0].annotate(
    f'Loss finale : {loss_history[-1]:.4f}',
    xy=(len(loss_history)-1, loss_history[-1]),
    xytext=(len(loss_history)*0.6, loss_history[0]*0.7),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10
)

# --- Graphique 2 : Courbe d'accuracy ---
axes[1].plot(
    [a*100 for a in acc_history],
    color='#185FA5', linewidth=2, label='Accuracy (train)'
)
axes[1].axhline(
    y=acc_test_final, color='#0F6E56',
    linewidth=1.5, linestyle='--',
    label=f'Accuracy test finale : {acc_test_final:.1f}%'
)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title("Evolution de l'accuracy", fontsize=12)
axes[1].set_ylim([0, 105])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./courbe_loss_accuracy.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphiques sauvegardes : courbe_loss_accuracy.png")


Jour 8: comparaison

Preparation commune des donnees

In [ ]:
print("=" * 60)
print("PREPARATION DES DONNEES (commune aux deux modeles)")
print("=" * 60)

iris = load_iris()
X_raw = iris.data
y_raw = iris.target

# Split identique pour les deux modeles (meme random_state)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Normalisation identique pour les deux modeles
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# Format reseau from scratch : (n_features, m)
X_train_fs = X_train_scaled.T
X_test_fs  = X_test_scaled.T
Y_train_fs = one_hot_encode(y_train, n_classes=3)
Y_test_fs  = one_hot_encode(y_test,  n_classes=3)

# Format sklearn : (m, n_features) — sklearn prefere les lignes = exemples
X_train_sk = X_train_scaled
X_test_sk  = X_test_scaled
y_train_sk = y_train
y_test_sk  = y_test

print(f"\nDataset Iris : 150 exemples, 4 features, 3 classes")
print(f"Train : {X_train_sk.shape[0]} exemples | Test : {X_test_sk.shape[0]} exemples")
print(f"Normalisation : StandardScaler (moyenne=0, std=1)")

Entrainement du reseau FROM SCRATCH

In [ ]:
print("\n" + "=" * 60)
print("MODELE 1 : Reseau from scratch (NumPy)")
print("=" * 60)

# Architecture : 2 couches cachees [4, 16, 8, 3]
# Learning rate : 0.05 | Iterations : 2000
layer_dims = [4, 16, 8, 3]
LEARNING_RATE = 0.05
NUM_ITERATIONS = 2000

print(f"\nArchitecture : {layer_dims}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Iterations : {NUM_ITERATIONS}")
print()

params_fs, loss_history_fs, acc_history_fs, temps_fs = model(
    X_train_fs, Y_train_fs,
    X_test_fs,  Y_test_fs,
    layer_dims        = layer_dims,
    learning_rate     = LEARNING_RATE,
    num_iterations    = NUM_ITERATIONS,
    print_every       = 500,
    output_activation = 'softmax',
    verbose           = True
)

# Resultats from scratch
acc_train_fs = compute_accuracy(X_train_fs, Y_train_fs, params_fs) * 100
acc_test_fs  = compute_accuracy(X_test_fs,  Y_test_fs,  params_fs) * 100
y_pred_fs    = predict(X_test_fs, params_fs)
loss_finale_fs = loss_history_fs[-1]

print(f"\nResultats from scratch :")
print(f"  Accuracy TRAIN : {acc_train_fs:.1f}%")
print(f"  Accuracy TEST  : {acc_test_fs:.1f}%")
print(f"  Loss finale    : {loss_finale_fs:.4f}")
print(f"  Temps          : {temps_fs:.3f} secondes")

Entrainement du MLPClassifier sklearn

In [ ]:
#Entrainement du MLPClassifier sklearn.
print("\n" + "=" * 60)
print("MODELE 2 : MLPClassifier (scikit-learn)")
print("=" * 60)

# On utilise la meme architecture et les memes hyperparametres
# hidden_layer_sizes=(16, 8) = 2 couches cachees de 16 et 8 neurones
# activation='relu'          = ReLU dans les couches cachees
# solver='sgd'               = descente de gradient stochastique
# learning_rate_init=0.05    = meme learning rate
# max_iter=2000              = meme nombre d'iterations
# random_state=42            = reproductibilite

mlp_sklearn = MLPClassifier(
    hidden_layer_sizes = (16, 8),   # meme architecture que from scratch
    activation         = 'relu',    # meme activation dans couches cachees
    solver             = 'sgd',     # meme optimiseur (SGD)
    learning_rate_init = LEARNING_RATE,
    max_iter           = NUM_ITERATIONS,
    random_state       = 42,
    verbose            = False,
    n_iter_no_change   = NUM_ITERATIONS,  # desactive l'arret precoce
    tol                = 1e-10
)

print(f"\nArchitecture : [4, 16, 8, 3]")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Iterations : {NUM_ITERATIONS}")
print(f"\nEntrainement en cours...")

# Mesure du temps d'entrainement sklearn
start_sk = time.time()
mlp_sklearn.fit(X_train_sk, y_train_sk)
temps_sk = time.time() - start_sk

# Resultats sklearn
acc_train_sk = mlp_sklearn.score(X_train_sk, y_train_sk) * 100
acc_test_sk  = mlp_sklearn.score(X_test_sk,  y_test_sk)  * 100
y_pred_sk    = mlp_sklearn.predict(X_test_sk)
loss_finale_sk = mlp_sklearn.loss_

print(f"\nResultats sklearn :")
print(f"  Accuracy TRAIN : {acc_train_sk:.1f}%")
print(f"  Accuracy TEST  : {acc_test_sk:.1f}%")
print(f"  Loss finale    : {loss_finale_sk:.4f}")
print(f"  Temps          : {temps_sk:.3f} secondes")
print(f"  Iterations reelles effectuees : {mlp_sklearn.n_iter_}")


Tableau comparatif

In [ ]:
print("\n" + "=" * 60)
print("TABLEAU COMPARATIF")
print("=" * 60)

# Calcul du nombre de parametres
def count_parameters(layer_dims):
    """Compte le nombre total de parametres (W + b) du reseau."""
    total = 0
    for l in range(1, len(layer_dims)):
        total += layer_dims[l] * layer_dims[l-1]  # poids W
        total += layer_dims[l]                      # biais b
    return total


nb_params = count_parameters(layer_dims)

import pandas as pd

# Construction du tableau comparatif avec pandas
data = {
    'Critere': [
        'Architecture',
        'Accuracy TRAIN',
        'Accuracy TEST',
        'Loss finale',
        'Temps entrainement',
        'Nb de parametres',
        'Framework',
        'Lignes de code (approx.)'
    ],
    'From Scratch (NumPy)': [
        str(layer_dims),
        f'{acc_train_fs:.1f}%',
        f'{acc_test_fs:.1f}%',
        f'{loss_finale_fs:.4f}',
        f'{temps_fs:.3f} s',
        str(nb_params),
        'NumPy only',
        '~150 lignes'
    ],
    'scikit-learn': [
        str(layer_dims),
        f'{acc_train_sk:.1f}%',
        f'{acc_test_sk:.1f}%',
        f'{loss_finale_sk:.4f}',
        f'{temps_sk:.3f} s',
        str(nb_params),
        'scikit-learn',
        '~10 lignes'
    ]
}

df = pd.DataFrame(data)

# Affichage stylise
display(df.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'padding': '8px 16px'
    })
    .set_properties(subset=['Critere'], **{
        'text-align': 'left',
        'font-weight': 'bold',
        'background-color': '#1e3a5f',
        'color': 'white'
    })
    .set_properties(subset=['From Scratch (NumPy)'], **{
        'background-color': '#fdf0f0',
        'color': '#A32D2D'
    })
    .set_properties(subset=['scikit-learn'], **{
        'background-color': '#f0f5ff',
        'color': '#185FA5'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#1e3a5f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('text-align', 'center'),
            ('padding', '10px 16px'),
            ('border', '1px solid #ddd')
        ]
    }])
    .hide(axis='index')
)

print("Analyse :")
diff_test = abs(acc_test_fs - acc_test_sk)
print(f"  -> Difference d'accuracy sur le test : {diff_test:.1f}%")
if diff_test <= 5:
    print("  -> Les deux modeles sont comparables en performance.")
    print("  -> La valeur du from scratch : comprendre chaque calcul.")
ratio_temps = temps_sk / temps_fs if temps_fs > 0 else 1
print(f"  -> sklearn est {ratio_temps:.1f}x plus {'rapide' if ratio_temps < 1 else 'lent'} "
      f"que from scratch sur ce dataset.")
print("  -> sklearn est plus optimise (C, Cython) mais boite noire.")
print("  -> From scratch : transparence totale, apprentissage profond.")


Visualisations comparatives

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Comparaison : Reseau from scratch vs scikit-learn MLPClassifier\n'
    'Dataset Iris | Architecture [4, 16, 8, 3] | lr=0.05 | 2000 iterations',
    fontsize=12, fontweight='bold'
)

# --- Graphique 1 : Courbes de loss ---
axes[0].plot(
    loss_history_fs, color='#A32D2D',
    linewidth=2, label='From scratch (NumPy)'
)
# sklearn stocke la loss par iteration dans loss_curve_
if hasattr(mlp_sklearn, 'loss_curve_'):
    axes[0].plot(
        mlp_sklearn.loss_curve_, color='#185FA5',
        linewidth=2, linestyle='--', label='scikit-learn'
    )
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Graphique 2 : Barres comparatives accuracy ---
modeles  = ['From Scratch\n(NumPy)', 'scikit-learn\n(MLPClassifier)']
acc_train = [acc_train_fs, acc_train_sk]
acc_test  = [acc_test_fs,  acc_test_sk]
x = np.arange(len(modeles))
width = 0.35

bars1 = axes[1].bar(x - width/2, acc_train, width,
                     label='Train', color='#A32D2D', alpha=0.85)
bars2 = axes[1].bar(x + width/2, acc_test,  width,
                     label='Test',  color='#185FA5', alpha=0.85)

# Ajouter les valeurs sur les barres
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Train vs Test')
axes[1].set_xticks(x)
axes[1].set_xticklabels(modeles)
axes[1].set_ylim([85, 105])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# --- Graphique 3 : Matrices de confusion cote a cote ---
from sklearn.metrics import ConfusionMatrixDisplay

# Matrice from scratch
cm_fs = confusion_matrix(y_test, y_pred_fs)
disp_fs = ConfusionMatrixDisplay(
    confusion_matrix=cm_fs,
    display_labels=iris.target_names
)
disp_fs.plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(
    f'Matrice de confusion\nFrom Scratch — Test acc : {acc_test_fs:.1f}%'
)

plt.tight_layout()
plt.savefig(
    './comparaison_from_scratch_vs_sklearn.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("\nGraphique sauvegarde : comparaison_from_scratch_vs_sklearn.png")


Matrice de confusion sklearn separee

In [ ]:
fig2, ax = plt.subplots(figsize=(6, 5))
cm_sk = confusion_matrix(y_test, y_pred_sk)
disp_sk = ConfusionMatrixDisplay(
    confusion_matrix=cm_sk,
    display_labels=iris.target_names
)
disp_sk.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title(
    f'Matrice de confusion\nscikit-learn — Test acc : {acc_test_sk:.1f}%'
)
plt.tight_layout()
plt.savefig(
    './matrice_confusion_sklearn.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Graphique sauvegarde : matrice_confusion_sklearn.png")

print("Les deux modeles sont implementes et compares.")

PREPARATION DES DONNEES

Analyse 1 : Impact du Learning Rate

In [ ]:
print("=" * 55)
print("ANALYSE 1 : Impact du Learning Rate")
print("=" * 55)

# On teste 5 learning rates differents
# Tout le reste est identique : meme architecture, meme nb d'iterations
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5]
layer_dims_lr  = [4, 16, 8, 3]

results_lr = {}   # stocke les resultats pour chaque lr

for lr in learning_rates:
    np.random.seed(42)   # meme initialisation pour comparaison equitable
    params, loss_hist, acc_hist, temps = model(
        X_train, Y_train, X_test, Y_test,
        layer_dims     = layer_dims_lr,
        learning_rate  = lr,
        num_iterations = NUM_ITERATIONS,
        verbose        = False   # silencieux : on teste beaucoup de configs
    )
    acc_test_final = compute_accuracy(X_test, Y_test, params) * 100
    results_lr[lr] = {
        'loss_history' : loss_hist,
        'acc_test'     : acc_test_final,
        'loss_finale'  : loss_hist[-1],
        'temps'        : temps
    }
    print(f"  lr={lr:.3f} | Loss finale : {loss_hist[-1]:.4f} | "
          f"Acc test : {acc_test_final:.1f}%")

# --- Graphique Learning Rate ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Analyse 1 : Impact du Learning Rate\n"
    f"Architecture {layer_dims_lr} | {NUM_ITERATIONS} iterations",
    fontsize=13, fontweight='bold'
)

colors = ['#E24B4A', '#BA7517', '#0F6E56', '#185FA5', '#534AB7']

# Courbes de loss
for i, lr in enumerate(learning_rates):
    axes[0].plot(
        results_lr[lr]['loss_history'],
        label=f'lr={lr}',
        color=colors[i], linewidth=1.8
    )
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la Loss')
axes[0].legend()
axes[0].set_ylim([0, 2])
axes[0].grid(True, alpha=0.3)

# Barres d'accuracy finale
acc_values = [results_lr[lr]['acc_test'] for lr in learning_rates]
bars = axes[1].bar(
    [str(lr) for lr in learning_rates],
    acc_values,
    color=colors, alpha=0.85, edgecolor='white', linewidth=0.5
)
for bar, val in zip(bars, acc_values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Accuracy TEST (%)')
axes[1].set_title('Accuracy finale par Learning Rate')
axes[1].set_ylim([0, 110])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./analyse_learning_rate.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphique sauvegarde : analyse_learning_rate.png")


Analyse 2 : Impact de l'Architecture

In [ ]:
print("\n" + "=" * 55)
print("ANALYSE 2 : Impact de l'Architecture")
print("=" * 55)

# On teste 6 architectures differentes
# Entree=4 (Iris), Sortie=3 (3 classes) — on varie les couches cachees
architectures = [
    [4, 4, 3],          # reseau tres petit
    [4, 8, 3],          # petit
    [4, 16, 3],         # moyen 1 couche
    [4, 16, 8, 3],      # moyen 2 couches (notre reference)
    [4, 32, 16, 3],     # grand
    [4, 64, 32, 16, 3], # tres grand
]

results_arch = {}

for arch in architectures:
    np.random.seed(42)
    params, loss_hist, acc_hist, temps = model(
        X_train, Y_train, X_test, Y_test,
        layer_dims     = arch,
        learning_rate  = 0.05,
        num_iterations = NUM_ITERATIONS,
        verbose        = False
    )
    acc_test  = compute_accuracy(X_test,  Y_test,  params) * 100
    acc_train = compute_accuracy(X_train, Y_train, params) * 100

    # Calcul du nombre de parametres
    nb_params = sum(
        arch[l]*arch[l-1] + arch[l]
        for l in range(1, len(arch))
    )

    arch_str = str(arch)
    results_arch[arch_str] = {
        'loss_history' : loss_hist,
        'acc_test'     : acc_test,
        'acc_train'    : acc_train,
        'loss_finale'  : loss_hist[-1],
        'nb_params'    : nb_params,
        'temps'        : temps
    }
    print(f"  {arch_str:25s} | Params : {nb_params:4d} | "
          f"Acc train : {acc_train:.1f}% | Acc test : {acc_test:.1f}%")

# --- Graphique Architecture ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    "Analyse 2 : Impact de l'Architecture\n"
    f"lr=0.05 | {NUM_ITERATIONS} iterations",
    fontsize=13, fontweight='bold'
)

colors_arch = ['#E24B4A','#BA7517','#0F6E56','#185FA5','#534AB7','#8B4513']
arch_labels = [str(a) for a in architectures]

# Courbes de loss
for i, arch in enumerate(architectures):
    arch_str = str(arch)
    axes[0].plot(
        results_arch[arch_str]['loss_history'],
        label=arch_str,
        color=colors_arch[i], linewidth=1.8
    )
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la Loss par Architecture')
axes[0].legend(fontsize=8)
axes[0].set_ylim([0, 1.5])
axes[0].grid(True, alpha=0.3)

# Comparaison train vs test (detection overfitting)
x = np.arange(len(architectures))
width = 0.35
acc_trains = [results_arch[str(a)]['acc_train'] for a in architectures]
acc_tests  = [results_arch[str(a)]['acc_test']  for a in architectures]

bars1 = axes[1].bar(x - width/2, acc_trains, width,
                     label='Train', color='#A32D2D', alpha=0.8)
bars2 = axes[1].bar(x + width/2, acc_tests,  width,
                     label='Test',  color='#185FA5', alpha=0.8)

for bar in bars1:
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.3,
                 f'{bar.get_height():.0f}%',
                 ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar in bars2:
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.3,
                 f'{bar.get_height():.0f}%',
                 ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[1].set_xticks(x)
axes[1].set_xticklabels(
    [str(a) for a in architectures],
    rotation=15, ha='right', fontsize=8
)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Train vs Test par Architecture')
axes[1].set_ylim([80, 110])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('./analyse_architecture.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphique sauvegarde : analyse_architecture.png")


Analyse 3 : Underfitting vs Overfitting

In [ ]:
print("\n" + "=" * 55)
print("ANALYSE 3 : Detection Underfitting / Overfitting")
print("=" * 55)

# On entraine avec differents nombres d'iterations
# pour voir comment evolue le gap train/test
iterations_list = [50, 100, 200, 500, 1000, 2000, 5000]
layer_dims_of   = [4, 32, 16, 3]

acc_trains_of = []
acc_tests_of  = []

for n_iter in iterations_list:
    np.random.seed(42)
    params, _, _, _ = model(
        X_train, Y_train, X_test, Y_test,
        layer_dims     = layer_dims_of,
        learning_rate  = 0.05,
        num_iterations = n_iter,
        verbose        = False
    )
    acc_trains_of.append(compute_accuracy(X_train, Y_train, params) * 100)
    acc_tests_of.append( compute_accuracy(X_test,  Y_test,  params) * 100)
    print(f"  {n_iter:5d} iterations | "
          f"Acc train : {acc_trains_of[-1]:.1f}% | "
          f"Acc test  : {acc_tests_of[-1]:.1f}%")

# --- Graphique Underfitting / Overfitting ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iterations_list, acc_trains_of,
        'o-', color='#A32D2D', linewidth=2, label='Accuracy TRAIN')
ax.plot(iterations_list, acc_tests_of,
        's--', color='#185FA5', linewidth=2, label='Accuracy TEST')

# Zones annotees
ax.axvspan(0, 200, alpha=0.08, color='red',
           label='Zone Underfitting (trop peu iterations)')
ax.axvspan(200, 5000, alpha=0.05, color='green',
           label='Zone apprentissage correct')

ax.set_xscale('log')
ax.set_xlabel("Nombre d'iterations (echelle log)", fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(
    f"Underfitting vs Overfitting\nArchitecture {layer_dims_of} | lr=0.05",
    fontsize=13, fontweight='bold'
)
ax.set_ylim([30, 105])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Annotations texte
ax.annotate('Underfitting\n(pas assez appris)',
            xy=(100, acc_trains_of[2]),
            xytext=(60, 55),
            fontsize=9, color='#A32D2D',
            arrowprops=dict(arrowstyle='->', color='#A32D2D'))
ax.annotate('Bon equilibre\ntrain ≈ test',
            xy=(1000, acc_tests_of[5]),
            xytext=(800, 80),
            fontsize=9, color='#0F6E56',
            arrowprops=dict(arrowstyle='->', color='#0F6E56'))

plt.tight_layout()
plt.savefig('./underfitting_overfitting.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphique sauvegarde : underfitting_overfitting.png")


Tableau recapitulatif des analyses

In [ ]:
print("\n" + "=" * 55)
print("TABLEAU RECAPITULATIF — Meilleures configurations")
print("=" * 55)

# Meilleur learning rate
best_lr  = max(results_lr,  key=lambda lr: results_lr[lr]['acc_test'])
# Meilleure architecture
best_arch = max(results_arch, key=lambda a: results_arch[a]['acc_test'])

data_recap = {
    'Analyse': [
        'Meilleur Learning Rate',
        'Meilleure Architecture',
        'Configuration reference'
    ],
    'Valeur': [
        str(best_lr),
        best_arch,
        'lr=0.05, [4,16,8,3]'
    ],
    'Acc TEST': [
        f"{results_lr[best_lr]['acc_test']:.1f}%",
        f"{results_arch[best_arch]['acc_test']:.1f}%",
        '96.7%'
    ],
    'Remarque': [
        'Convergence rapide et stable',
        'Meilleur compromis params/perf',
        'Utilisee pour comparaison sklearn'
    ]
}

df_recap = pd.DataFrame(data_recap)

display(df_recap.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'padding': '8px 14px'
    })
    .set_properties(subset=['Analyse'], **{
        'text-align': 'left',
        'font-weight': 'bold',
        'background-color': '#1e3a5f',
        'color': 'white'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#1e3a5f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('padding', '10px 14px'),
            ('text-align', 'center')
        ]
    }])
    .hide(axis='index')
)

APPROFONDISSEMENT

APPLICATION AUX IMAGES

 Chargement et preparation de MNIST

In [ ]:
# Chargement et preparation de MNIST
print("=" * 60)
print("ETAPE 1 : Chargement du dataset MNIST")
print("=" * 60)
print("\nTelechargementen cours (peut prendre 1-2 minutes)...")

# Chargement via sklearn
# fetch_openml retourne deja les images aplaties en (70000, 784)
# Plus besoin de flatten manuel !
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X_raw = mnist.data                    # shape : (70000, 784), valeurs 0-255
y_raw = mnist.target.astype(int)     # shape : (70000,), labels 0-9

print(f"\nDataset MNIST charge :")
print(f"  X_raw shape : {X_raw.shape}   (70000 images, 784 pixels chacune)")
print(f"  y_raw shape : {y_raw.shape}   (70000 labels, valeurs 0-9)")
print(f"  Type pixels : {X_raw.dtype}   (entiers 0-255)")
print(f"  Classes     : {np.unique(y_raw)}")

# Verification de la repartition des classes
print(f"\nRepartition des classes :")
for c in range(10):
    n = np.sum(y_raw == c)
    print(f"  Chiffre {c} : {n} exemples ({n/len(y_raw)*100:.1f}%)")


In [ ]:
#Preprocessing : normalisation et split
print("\n" + "=" * 60)
print("ETAPE 2 : Preprocessing")
print("=" * 60)

# --- Split train/test ---
# On garde 10 000 exemples pour le test (standard MNIST)
# stratify garantit la meme proportion de chaque chiffre
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw,
    test_size  = 10000,
    random_state = 42,
    stratify   = y_raw
)

print(f"\nApres split :")
print(f"  X_train_raw shape : {X_train_raw.shape}  (60000 images)")
print(f"  X_test_raw shape  : {X_test_raw.shape}   (10000 images)")

# --- Normalisation : diviser par 255 ---
# Ramene les valeurs de [0, 255] vers [0.0, 1.0]
# Stabilise le gradient pendant la backprop
X_train_norm = X_train_raw / 255.0
X_test_norm  = X_test_raw  / 255.0

print(f"\nApres normalisation (/ 255) :")
print(f"  Valeur min : {X_train_norm.min():.3f}")
print(f"  Valeur max : {X_train_norm.max():.3f}")
print(f"  Valeur moy : {X_train_norm.mean():.3f}")

# --- Transposition : (m, 784) -> (784, m) ---
# Notre reseau attend shape (n_x, m) pas (m, n_x)
X_train = X_train_norm.T    # shape : (784, 60000)
X_test  = X_test_norm.T     # shape : (784, 10000)

# --- One-Hot Encoding des labels ---
# y : [3, 7, 0, ...] -> Y : matrice (10, m) avec 1 a la bonne ligne
Y_train = one_hot_encode(y_train, n_classes=10)  # shape : (10, 60000)
Y_test  = one_hot_encode(y_test,  n_classes=10)  # shape : (10, 10000)

print(f"\nApres transposition et one-hot :")
print(f"  X_train shape : {X_train.shape}")
print(f"  Y_train shape : {Y_train.shape}")
print(f"  X_test shape  : {X_test.shape}")
print(f"  Y_test shape  : {Y_test.shape}")



##Visualisation des images MNIST

In [ ]:
print("\n" + "=" * 60)
print("ETAPE 3 : Visualisation d'exemples MNIST")
print("=" * 60)

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
fig.suptitle(
    'Exemples du dataset MNIST — 3 exemples par chiffre (0 a 9)',
    fontsize=13, fontweight='bold'
)

for digit in range(10):
    # Trouver 3 exemples de ce chiffre dans le train
    indices = np.where(y_train == digit)[0][:3]
    for row, idx in enumerate(indices):
        ax = axes[row, digit]
        # Redonner la forme 28x28 pour afficher
        # X_train est (784, m), donc on prend la colonne idx
        # et on reshape en (28, 28)
        img = X_train[:, idx].reshape(28, 28)
        ax.imshow(img, cmap='gray_r')  # gray_r : fond blanc, encre noire
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('./mnist_exemples.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nImage sauvegardee : mnist_exemples.png")


Entrainement du reseau from scratch sur MNIST

In [ ]:
print("\n" + "=" * 60)
print("ETAPE 4 : Entrainement du reseau from scratch")
print("=" * 60)

# Architecture : [784, 128, 64, 10]
# 784 = nb de pixels MNIST
# 128, 64 = couches cachees (ReLU)
# 10 = nb de classes (chiffres 0-9)
layer_dims_mnist = [784, 128, 64, 10]

# On utilise un sous-ensemble pour aller plus vite
# 10 000 exemples suffisent pour obtenir de bons resultats rapidement
# Tu peux augmenter a 60000 pour un meilleur resultat final
N_TRAIN = 10000
X_train_sub = X_train[:, :N_TRAIN]
Y_train_sub = Y_train[:, :N_TRAIN]

print(f"\nArchitecture : {layer_dims_mnist}")
print(f"Exemples train utilises : {N_TRAIN} / 60000")
print(f"Learning rate : 0.1")
print(f"Iterations : 300")
print()

params_mnist, loss_hist_mnist, acc_hist_mnist, temps_mnist = model(
    X_train_sub, Y_train_sub,
    X_test,      Y_test,
    layer_dims        = layer_dims_mnist,
    learning_rate     = 0.1,
    num_iterations    = 300,
    print_every       = 50,
    output_activation = 'softmax',
    verbose           = True
)

# Resultats finaux
acc_train_mnist = compute_accuracy(
    X_train_sub, Y_train_sub, params_mnist
) * 100
acc_test_mnist  = compute_accuracy(
    X_test, Y_test, params_mnist
) * 100

print(f"\nResultats finaux (from scratch) :")
print(f"  Accuracy TRAIN : {acc_train_mnist:.1f}%")
print(f"  Accuracy TEST  : {acc_test_mnist:.1f}%")
print(f"  Loss finale    : {loss_hist_mnist[-1]:.4f}")
print(f"  Temps          : {temps_mnist:.1f} secondes")


Comparaison avec MLPClassifier sklearn

In [ ]:
print("\n" + "=" * 60)
print("ETAPE 5 : Comparaison avec scikit-learn")
print("=" * 60)

# Meme architecture, meme learning rate, meme nb d'iterations
mlp_mnist = MLPClassifier(
    hidden_layer_sizes = (128, 64),
    activation         = 'relu',
    solver             = 'sgd',
    learning_rate_init = 0.1,
    max_iter           = 300,
    random_state       = 42,
    verbose            = False,
    n_iter_no_change   = 300,
    tol                = 1e-10
)

print(f"\nEntrainement sklearn sur {N_TRAIN} exemples...")
start_sk = time.time()
mlp_mnist.fit(X_train_sub.T, y_train[:N_TRAIN])
temps_sk_mnist = time.time() - start_sk

acc_train_sk_mnist = mlp_mnist.score(X_train_sub.T, y_train[:N_TRAIN]) * 100
acc_test_sk_mnist  = mlp_mnist.score(X_test.T,      y_test) * 100

print(f"\nResultats sklearn :")
print(f"  Accuracy TRAIN : {acc_train_sk_mnist:.1f}%")
print(f"  Accuracy TEST  : {acc_test_sk_mnist:.1f}%")
print(f"  Temps          : {temps_sk_mnist:.1f} secondes")


Tableau comparatif MNIST

In [ ]:
print("\n" + "=" * 60)
print("TABLEAU COMPARATIF MNIST")
print("=" * 60)

nb_params_mnist = sum(
    layer_dims_mnist[l] * layer_dims_mnist[l-1] + layer_dims_mnist[l]
    for l in range(1, len(layer_dims_mnist))
)

data_mnist = {
    'Critere': [
        'Dataset',
        'Architecture',
        'Exemples train',
        'Accuracy TRAIN',
        'Accuracy TEST',
        'Loss finale',
        'Temps entrainement',
        'Nb de parametres',
        'Framework'
    ],
    'From Scratch (NumPy)': [
        'MNIST',
        str(layer_dims_mnist),
        str(N_TRAIN),
        f'{acc_train_mnist:.1f}%',
        f'{acc_test_mnist:.1f}%',
        f'{loss_hist_mnist[-1]:.4f}',
        f'{temps_mnist:.1f} s',
        str(nb_params_mnist),
        'NumPy only'
    ],
    'scikit-learn': [
        'MNIST',
        str(layer_dims_mnist),
        str(N_TRAIN),
        f'{acc_train_sk_mnist:.1f}%',
        f'{acc_test_sk_mnist:.1f}%',
        f'{mlp_mnist.loss_:.4f}',
        f'{temps_sk_mnist:.1f} s',
        str(nb_params_mnist),
        'scikit-learn'
    ]
}

df_mnist = pd.DataFrame(data_mnist)
display(df_mnist.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'padding': '8px 16px'
    })
    .set_properties(subset=['Critere'], **{
        'text-align': 'left',
        'font-weight': 'bold',
        'background-color': '#1e3a5f',
        'color': 'white'
    })
    .set_properties(subset=['From Scratch (NumPy)'], **{
        'background-color': '#fdf0f0',
        'color': '#A32D2D'
    })
    .set_properties(subset=['scikit-learn'], **{
        'background-color': '#f0f5ff',
        'color': '#185FA5'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#1e3a5f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('text-align', 'center'),
            ('padding', '10px 16px')
        ]
    }])
    .hide(axis='index')
)


Visualisations : courbes et matrice de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Reseau from scratch sur MNIST\n'
    f'Architecture {layer_dims_mnist} | lr=0.1 | {N_TRAIN} exemples',
    fontsize=13, fontweight='bold'
)

# Courbe de loss
axes[0].plot(loss_hist_mnist, color='#A32D2D', linewidth=2)
if hasattr(mlp_mnist, 'loss_curve_'):
    axes[0].plot(
        mlp_mnist.loss_curve_, color='#185FA5',
        linewidth=2, linestyle='--', label='sklearn'
    )
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la Loss')
axes[0].legend(['From scratch', 'sklearn'])
axes[0].grid(True, alpha=0.3)

# Matrice de confusion (from scratch)
y_pred_mnist = predict(X_test, params_mnist)
cm_mnist = confusion_matrix(y_test, y_pred_mnist)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_mnist,
    display_labels=list(range(10))
)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(
    f'Matrice de confusion\nFrom Scratch — Test acc : {acc_test_mnist:.1f}%'
)

plt.tight_layout()
plt.savefig('./mnist_resultats.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde : mnist_resultats.png")


Visualisation des erreurs du reseau

In [ ]:
print("\n" + "=" * 60)
print("ETAPE 6 : Analyse des erreurs")
print("=" * 60)

# Trouver les exemples mal classes
y_pred_all = predict(X_test, params_mnist)
y_true_all = y_test
erreurs_idx = np.where(y_pred_all != y_true_all)[0]

print(f"\nNombre d'erreurs : {len(erreurs_idx)} / {len(y_test)}")
print(f"Accuracy : {(1 - len(erreurs_idx)/len(y_test))*100:.1f}%")

# Afficher les 20 premieres erreurs
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle(
    'Exemples mal classes par le reseau from scratch\n'
    'Titre = vraie classe | Sous-titre = classe predite',
    fontsize=12, fontweight='bold'
)

for i, ax in enumerate(axes.flatten()):
    if i >= len(erreurs_idx):
        ax.axis('off')
        continue
    idx = erreurs_idx[i]
    img = X_test[:, idx].reshape(28, 28)
    vraie  = y_true_all[idx]
    predite = y_pred_all[idx]
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'Vrai : {vraie}', fontsize=10, fontweight='bold')
    ax.set_xlabel(f'Predit : {predite}', fontsize=10, color='red')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('./mnist_erreurs.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde : mnist_erreurs.png")

print("\n" + "=" * 60)
print("MNIST COMPLETE !")
print(f"Accuracy finale : {acc_test_mnist:.1f}% (from scratch)")
print(f"Accuracy sklearn: {acc_test_sk_mnist:.1f}%")
print("=" * 60)


Chargement de CIFAR-10

In [ ]:
from tensorflow.keras.datasets import cifar10

CLASSES = [
    'avion', 'voiture', 'oiseau', 'chat', 'cerf',
    'chien', 'grenouille', 'cheval', 'bateau', 'camion'
]

# Chargement automatique via Keras (téléchargement la première fois)
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

# Aplatir les labels (de (N,1) à (N,))
y_train_raw = y_train_raw.flatten()
y_test_raw = y_test_raw.flatten()

print(f"\nCIFAR-10 charge avec succes depuis keras.datasets !")
print(f"  X_train shape : {X_train_raw.shape}  (50000 images, 32x32, RGB)")
print(f"  y_train shape : {y_train_raw.shape}")
print(f"  X_test shape  : {X_test_raw.shape}   (10000 images)")
print(f"  Classes       : {CLASSES}")

Visualisation des images CIFAR-10

In [ ]:
print("\n" + "=" * 60)
print("Visualisation des images CIFAR-10")
print("=" * 60)

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
fig.suptitle(
    'Exemples CIFAR-10 — 3 photos par classe\n'
    'Images couleur 32x32 pixels (RGB)',
    fontsize=13, fontweight='bold'
)

for classe in range(10):
    indices = np.where(y_train_raw == classe)[0][:3]
    for row, idx in enumerate(indices):
        ax = axes[row, classe]
        # CIFAR-10 : shape (32, 32, 3) — directement affichable
        # Pas besoin de reshape car les images sont deja en 2D+couleur
        ax.imshow(X_train_raw[idx])
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASSES[classe], fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('./cifar10_exemples.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nImage sauvegardee : cifar10_exemples.png")


Preprocessing CIFAR-10 pour le MLP

In [ ]:
print("\n" + "=" * 60)
print("Preprocessing CIFAR-10")
print("=" * 60)

# Nombre d'exemples pour l'entrainement
# On limite pour la rapidite (CIFAR est plus difficile)
N_TRAIN_SUB = 10000
N_TEST_SUB  = 2000

# --- Etape 1 : Aplatissement ---
# CIFAR : (m, 32, 32, 3) -> (m, 3072)
# 32 * 32 * 3 = 3072 valeurs par image (vs 784 pour MNIST)
# On deroule : ligne par ligne, puis canal R, puis G, puis B
X_train_flat = X_train_raw[:N_TRAIN_SUB].reshape(N_TRAIN_SUB, -1)
X_test_flat  = X_test_raw[:N_TEST_SUB].reshape(N_TEST_SUB, -1)

print(f"\nApres aplatissement :")
print(f"  (m, 32, 32, 3) -> (m, 3072)")
print(f"  X_train_flat shape : {X_train_flat.shape}")
print(f"  32 x 32 x 3 = {32*32*3} valeurs par image")

# --- Etape 2 : Normalisation ---
X_train_norm = X_train_flat / 255.0
X_test_norm  = X_test_flat  / 255.0

# --- Etape 3 : Transposition (m, 3072) -> (3072, m) ---
X_train = X_train_norm.T   # shape : (3072, N_TRAIN_SUB)
X_test  = X_test_norm.T    # shape : (3072, N_TEST_SUB)

# --- Etape 4 : Labels ---
y_train_sub = y_train_raw[:N_TRAIN_SUB]
y_test_sub  = y_test_raw[:N_TEST_SUB]
Y_train = one_hot_encode(y_train_sub, n_classes=10)
Y_test  = one_hot_encode(y_test_sub,  n_classes=10)

print(f"\nApres normalisation et transposition :")
print(f"  X_train shape : {X_train.shape}  (3072 features, {N_TRAIN_SUB} exemples)")
print(f"  Y_train shape : {Y_train.shape}  (10 classes, {N_TRAIN_SUB} exemples)")

print(f"\nComparaison avec MNIST :")
print(f"  MNIST  : 784 pixels/image  (28x28 gris)")
print(f"  CIFAR  : 3072 pixels/image (32x32x3 couleur)")
print(f"  -> 4x plus de donnees par image")
print(f"  -> Le reseau doit etre plus grand et plus de parametres")


Entrainement MLP from scratch sur CIFAR-10

In [ ]:
print("\n" + "=" * 60)
print("Entrainement MLP from scratch sur CIFAR-10")
print("=" * 60)

# Architecture adaptee a CIFAR-10
# 3072 entrees au lieu de 784 (MNIST) ou 4 (Iris)
layer_dims_cifar = [3072, 256, 128, 10]

print(f"\nArchitecture : {layer_dims_cifar}")
print(f"Exemples : {N_TRAIN_SUB} train, {N_TEST_SUB} test")
print(f"Learning rate : 0.01")
print(f"Iterations : 200")
print()

# Note : learning rate plus petit car CIFAR est plus difficile
# Trop grand -> divergence, trop petit -> lent
params_cifar, loss_hist_cifar, acc_hist_cifar, temps_cifar = model(
    X_train, Y_train,
    X_test,  Y_test,
    layer_dims        = layer_dims_cifar,
    learning_rate     = 0.01,
    num_iterations    = 200,
    print_every       = 40,
    output_activation = 'softmax',
    verbose           = True
)

acc_train_cifar = compute_accuracy(X_train, Y_train, params_cifar) * 100
acc_test_cifar  = compute_accuracy(X_test,  Y_test,  params_cifar) * 100

print(f"\nResultats MLP from scratch sur CIFAR-10 :")
print(f"  Accuracy TRAIN : {acc_train_cifar:.1f}%")
print(f"  Accuracy TEST  : {acc_test_cifar:.1f}%")
print(f"  (Rappel : hasard = 10%, MNIST MLP = 97%)")


Comparaison sklearn sur CIFAR-10

In [ ]:
print("\n" + "=" * 60)
print("Comparaison avec sklearn sur CIFAR-10")
print("=" * 60)

mlp_cifar = MLPClassifier(
    hidden_layer_sizes = (256, 128),
    activation         = 'relu',
    solver             = 'sgd',
    learning_rate_init = 0.01,
    max_iter           = 200,
    random_state       = 42,
    verbose            = False,
    n_iter_no_change   = 200,
    tol                = 1e-10
)

print("\nEntrainement sklearn...")
start_sk = time.time()
mlp_cifar.fit(X_train.T, y_train_sub)
temps_sk_cifar = time.time() - start_sk

acc_train_sk_cifar = mlp_cifar.score(X_train.T, y_train_sub) * 100
acc_test_sk_cifar  = mlp_cifar.score(X_test.T,  y_test_sub)  * 100

print(f"\nResultats sklearn sur CIFAR-10 :")
print(f"  Accuracy TRAIN : {acc_train_sk_cifar:.1f}%")
print(f"  Accuracy TEST  : {acc_test_sk_cifar:.1f}%")


Graphique de comparaison MNIST vs CIFAR-10

In [ ]:
print("\n" + "=" * 60)
print("Graphique : MNIST vs CIFAR-10 — Limite du MLP")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Limite du MLP from scratch : MNIST vs CIFAR-10\n'
    'Pourquoi les CNN ont ete inventes',
    fontsize=13, fontweight='bold'
)

# --- Graphique 1 : Courbes de loss comparees ---
axes[0].plot(loss_hist_cifar, color='#E24B4A', linewidth=2,
             label='CIFAR-10 (MLP)')
axes[0].axhline(y=np.log(10), color='gray', linewidth=1,
                linestyle=':', label=f'Loss hasard = {np.log(10):.2f}')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Convergence sur CIFAR-10')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].text(0.98, 0.98,
             f'Acc test finale\n{acc_test_cifar:.1f}%',
             transform=axes[0].transAxes,
             ha='right', va='top', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='#fdf0f0', alpha=0.8))

# --- Graphique 2 : Barres comparatives MNIST vs CIFAR ---
datasets    = ['MNIST\n(chiffres\nniveaux de gris)',
               'CIFAR-10\n(photos\ncouleur)']
acc_mlp     = [97.0, acc_test_cifar]
acc_cnn_ref = [99.3, 93.0]   # reference CNN dans la litterature
acc_hasard  = [10.0, 10.0]

x = np.arange(len(datasets))
w = 0.25

b1 = axes[1].bar(x - w, acc_mlp,     w, label='MLP from scratch',
                 color='#A32D2D', alpha=0.85)
b2 = axes[1].bar(x,     acc_cnn_ref, w, label='CNN (reference)',
                 color='#0F6E56', alpha=0.85)
b3 = axes[1].bar(x + w, acc_hasard,  w, label='Hasard (10 classes)',
                 color='#888780', alpha=0.6)

for bar in [*b1, *b2, *b3]:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 h + 0.5, f'{h:.0f}%',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_xticks(x)
axes[1].set_xticklabels(datasets, fontsize=10)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('MLP vs CNN vs Hasard')
axes[1].set_ylim([0, 115])
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

# --- Graphique 3 : Pourquoi CIFAR est plus difficile ---
categories = [
    'Pixels\n/image',
    'Canaux\ncouleur',
    'Variabilite\ndes objets',
    'Complexite\nglobale'
]
scores_mnist = [1, 1, 1, 1]    # reference = 1
scores_cifar = [4, 3, 8, 7]    # relatif a MNIST

angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

scores_mnist = scores_mnist + scores_mnist[:1]
scores_cifar = scores_cifar + scores_cifar[:1]

ax3 = axes[2]
ax3.remove()
ax3 = fig.add_subplot(133, projection='polar')

ax3.plot(angles, scores_mnist, 'o-', linewidth=2,
         color='#185FA5', label='MNIST')
ax3.fill(angles, scores_mnist, alpha=0.15, color='#185FA5')
ax3.plot(angles, scores_cifar, 'o-', linewidth=2,
         color='#A32D2D', label='CIFAR-10')
ax3.fill(angles, scores_cifar, alpha=0.15, color='#A32D2D')

ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(categories, fontsize=9)
ax3.set_ylim(0, 10)
ax3.set_title('Complexite relative\n(MNIST=1)', fontsize=11, pad=15)
ax3.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./mnist_vs_cifar10.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphique sauvegarde : mnist_vs_cifar10.png")


Tableau recapitulatif et conclusion

In [ ]:
print("\n" + "=" * 60)
print("TABLEAU RECAPITULATIF : Iris vs MNIST vs CIFAR-10")
print("=" * 60)

nb_params_cifar = sum(
    layer_dims_cifar[l] * layer_dims_cifar[l-1] + layer_dims_cifar[l]
    for l in range(1, len(layer_dims_cifar))
)

data_recap = {
    'Critere': [
        'Type de donnees',
        'Taille entree',
        'Nb classes',
        'Nb exemples train',
        'Accuracy MLP',
        'Accuracy CNN (ref)',
        'Architecture MLP',
        'Nb parametres',
        'Difficulte'
    ],
    'Iris': [
        'Numerique tabule',
        '4 features',
        '3',
        '120',
        '96.7%',
        '~96.7%',
        '[4,16,8,3]',
        '243',
        'Facile'
    ],
    'MNIST': [
        'Images gris',
        '784 pixels',
        '10',
        '60 000',
        '~97%',
        '~99.3%',
        '[784,128,64,10]',
        '109 386',
        'Moyen'
    ],
    'CIFAR-10': [
        'Photos couleur',
        '3072 pixels',
        '10',
        '50 000',
        f'~{acc_test_cifar:.0f}%',
        '~93%',
        '[3072,256,128,10]',
        str(nb_params_cifar),
        'Difficile'
    ]
}

df_recap = pd.DataFrame(data_recap)
display(df_recap.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'padding': '8px 14px'
    })
    .set_properties(subset=['Critere'], **{
        'text-align': 'left',
        'font-weight': 'bold',
        'background-color': '#1e3a5f',
        'color': 'white'
    })
    .set_properties(subset=['CIFAR-10'], **{
        'background-color': '#fdf0f0',
        'color': '#A32D2D'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#1e3a5f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('text-align', 'center'),
            ('padding', '10px 14px')
        ]
    }])
    .hide(axis='index')
)

print(f"""
CONCLUSION — Pourquoi le MLP est insuffisant sur CIFAR-10 :

1. PERTE D'INFORMATION SPATIALE
   Le flatten transforme (32,32,3) en vecteur 1D.
   Le reseau ne sait plus que les pixels voisins sont lies.
   Un chien peut etre a gauche ou a droite -> meme objet,
   mais vecteurs completement differents pour le MLP.

2. TROP DE VARIABILITE
   Meme classe (chien) : couleurs, angles, fonds differents.
   Le MLP doit memoriser chaque variante -> overfitting.

3. LA SOLUTION : LES CNN
   Les CNN appliquent des filtres qui detectent les bords,
   textures, formes -> ils comprennent la structure 2D.
   Ils sont invariants aux translations (objet a gauche
   ou droite = meme detection).

   MLP sur CIFAR-10  : ~{acc_test_cifar:.0f}%
   CNN sur CIFAR-10  : ~93%
   Difference        : +{93-acc_test_cifar:.0f}% grace a la structure spatiale
""")

print("=" * 60)
print("CIFAR-10 COMPLETE !")
print("Conclusion : le MLP est limite sur les vraies photos.")
print("Les CNN sont la prochaine etape logique.")
print("=" * 60)


In [ ]:
print("=" * 60)
print("PREPARATION DES DONNEES POUR LE CNN")
print("=" * 60)

# --- Normalisation ---
# Keras attend des valeurs float32 entre 0 et 1
# X_train_raw est de shape (50000, 32, 32, 3), valeurs 0-255
X_train_cnn = X_train_raw.astype('float32') / 255.0
X_test_cnn  = X_test_raw.astype('float32')  / 255.0

# IMPORTANT : Keras attend shape (m, H, W, C)
# C'est-a-dire (nb_exemples, hauteur, largeur, canaux)
# Notre X_train_raw est deja dans ce format -> pas de transposition !
# Contrairement au MLP qui attendait (n_features, m)

print(f"\nFormat Keras (m, H, W, C) :")
print(f"  X_train_cnn shape : {X_train_cnn.shape}")
print(f"  X_test_cnn shape  : {X_test_cnn.shape}")

# --- Labels : one-hot encoding avec Keras ---
# to_categorical convertit [3, 7, 0, ...] en matrice (m, 10)
# C'est l'equivalent de notre one_hot_encode() mais via Keras
Y_train_cnn = to_categorical(y_train_raw, num_classes=10)
Y_test_cnn  = to_categorical(y_test_raw,  num_classes=10)

print(f"\nLabels one-hot :")
print(f"  Y_train_cnn shape : {Y_train_cnn.shape}")
print(f"  Exemple y=3 -> {Y_train_cnn[np.where(y_train_raw==3)[0][0]]}")

# On limite a 10000 exemples pour aller plus vite
N = 10000
X_tr = X_train_cnn[:N]
Y_tr = Y_train_cnn[:N]
X_te = X_test_cnn
Y_te = Y_test_cnn

print(f"\nExemples utilises : {N} train, {len(X_te)} test")


 Architecture du CNN

In [ ]:
print("\n" + "=" * 60)
print("ARCHITECTURE DU CNN")
print("=" * 60)

# Construction du CNN avec Keras Sequential
# Sequential = on empile les couches les unes apres les autres

model_cnn = keras.Sequential([

    # --- BLOC 1 : Premiere convolution ---
    # Conv2D : applique 32 filtres de taille 3x3 sur l'image
    # activation='relu' : ReLU apres chaque convolution
    # padding='same' : garde la meme taille (32x32)
    # input_shape : (hauteur, largeur, canaux) = (32, 32, 3)
    layers.Conv2D(
        filters      = 32,          # 32 filtres differents
        kernel_size  = (3, 3),      # chaque filtre fait 3x3 pixels
        activation   = 'relu',      # ReLU apres convolution
        padding      = 'same',      # garde la taille 32x32
        input_shape  = (32, 32, 3)  # taille d'une image CIFAR
    ),

    # BatchNormalization : normalise les activations
    # Stabilise l'entrainement, permet un learning rate plus grand
    layers.BatchNormalization(),

    # Deuxieme convolution dans le meme bloc
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    # MaxPooling2D : reduit la taille de moitie (32x32 -> 16x16)
    # Prend le max dans chaque fenetre 2x2
    # Reduit le nombre de calculs et rend le reseau plus robuste
    layers.MaxPooling2D(pool_size=(2, 2)),  # 32x32 -> 16x16

    # Dropout : eteint aleatoirement 25% des neurones pendant l'entrainement
    # Evite l'overfitting (le reseau ne memorise pas les donnees)
    layers.Dropout(0.25),

    # --- BLOC 2 : Deuxieme convolution ---
    # 64 filtres maintenant : detecte des patterns plus complexes
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    # MaxPooling : 16x16 -> 8x8
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.25),

    # --- PARTIE DENSE (comme notre MLP) ---
    # Flatten : convertit les feature maps en vecteur 1D
    # 8x8x64 = 4096 valeurs -> vecteur de 4096
    layers.Flatten(),

    # Couche dense (comme dans notre MLP from scratch)
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # Couche de sortie : 10 classes, softmax
    layers.Dense(10, activation='softmax')
])

# Affichage du resume de l'architecture
model_cnn.summary()

# Calcul du nombre de parametres
nb_params_cnn = model_cnn.count_params()
print(f"\nNombre total de parametres : {nb_params_cnn:,}")


Compilation et entrainement

In [ ]:
print("\n" + "=" * 60)
print("COMPILATION ET ENTRAINEMENT DU CNN")
print("=" * 60)

# Compilation : definit comment le reseau va apprendre
model_cnn.compile(
    # Optimiseur Adam : plus efficace que SGD simple
    # Adapte le learning rate automatiquement
    optimizer = keras.optimizers.Adam(learning_rate=0.001),

    # Fonction de cout : cross-entropy categorielle
    # (equivalent a notre compute_loss cross_entropy_multi)
    loss = 'categorical_crossentropy',

    # Metrique a suivre pendant l'entrainement
    metrics = ['accuracy']
)

print(f"\nOptimiseur : Adam (lr=0.001)")
print(f"Loss       : categorical_crossentropy")
print(f"Exemples   : {N} train, {len(X_te)} test")
print(f"Epochs     : 15")
print(f"Batch size : 64")
print()

# Entrainement
# epochs : nombre de passages sur tout le dataset
# batch_size : nombre d'exemples traites a la fois
# validation_data : evalue sur le test apres chaque epoch
start_cnn = time.time()

history = model_cnn.fit(
    X_tr, Y_tr,
    epochs          = 15,
    batch_size      = 64,
    validation_data = (X_te, Y_te),
    verbose         = 1   # affiche la progression
)

temps_cnn = time.time() - start_cnn
print(f"\nTemps d'entrainement CNN : {temps_cnn:.1f} secondes")


Resultats et comparaison

In [ ]:
print("\n" + "=" * 60)
print("RESULTATS CNN vs MLP")
print("=" * 60)

# Evaluation finale du CNN
loss_cnn, acc_cnn = model_cnn.evaluate(X_te, Y_te, verbose=0)
acc_cnn_pct = acc_cnn * 100

# Predictions CNN
y_pred_cnn = np.argmax(model_cnn.predict(X_te, verbose=0), axis=1)

print(f"\nCNN Keras :")
print(f"  Accuracy TEST : {acc_cnn_pct:.1f}%")
print(f"  Loss finale   : {loss_cnn:.4f}")
print(f"  Temps         : {temps_cnn:.1f} s")


Tableau comparatif MLP vs CNN

In [ ]:
# Accuracy MLP from scratch sur CIFAR (recuperee du notebook precedent)
# Si la variable n'existe pas, on met une valeur par defaut
try:
    acc_mlp_cifar = acc_test_cifar
except NameError:
    acc_mlp_cifar = 45.0   # valeur typique MLP sur CIFAR-10

data_comp = {
    'Critere': [
        'Modele',
        'Dataset',
        'Architecture',
        'Accuracy TEST',
        'Loss finale',
        'Temps entrainement',
        'Nb parametres',
        'Structure spatiale',
        'Framework'
    ],
    'MLP from scratch': [
        'MLP (Dense only)',
        'CIFAR-10',
        '[3072, 256, 128, 10]',
        f'{acc_mlp_cifar:.1f}%',
        'N/A',
        'N/A',
        '~800 000',
        'Perdue (flatten)',
        'NumPy only'
    ],
    'CNN Keras': [
        'CNN (Conv + Dense)',
        'CIFAR-10',
        '2xConv32 + 2xConv64 + Dense512',
        f'{acc_cnn_pct:.1f}%',
        f'{loss_cnn:.4f}',
        f'{temps_cnn:.1f} s',
        f'{nb_params_cnn:,}',
        'Preservee (filtres)',
        'TensorFlow/Keras'
    ]
}

df_comp = pd.DataFrame(data_comp)
display(df_comp.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '13px',
        'border': '1px solid #ddd',
        'padding': '8px 16px'
    })
    .set_properties(subset=['Critere'], **{
        'text-align': 'left',
        'font-weight': 'bold',
        'background-color': '#1e3a5f',
        'color': 'white'
    })
    .set_properties(subset=['MLP from scratch'], **{
        'background-color': '#fdf0f0',
        'color': '#A32D2D'
    })
    .set_properties(subset=['CNN Keras'], **{
        'background-color': '#f0fff8',
        'color': '#0F6E56'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#1e3a5f'),
            ('color', 'white'),
            ('font-size', '13px'),
            ('text-align', 'center'),
            ('padding', '10px 16px')
        ]
    }])
    .hide(axis='index')
)


Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'CNN Keras vs MLP from scratch sur CIFAR-10\n'
    f'CNN Accuracy : {acc_cnn_pct:.1f}% | '
    f'MLP Accuracy : {acc_mlp_cifar:.1f}%',
    fontsize=13, fontweight='bold'
)

# --- Graphique 1 : Courbes d'accuracy CNN ---
axes[0].plot(history.history['accuracy'],
             color='#0F6E56', linewidth=2, label='Train')
axes[0].plot(history.history['val_accuracy'],
             color='#185FA5', linewidth=2,
             linestyle='--', label='Test')
axes[0].axhline(y=acc_mlp_cifar/100, color='#A32D2D',
                linewidth=1.5, linestyle=':',
                label=f'MLP ({acc_mlp_cifar:.0f}%)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy CNN par epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Graphique 2 : Courbes de loss CNN ---
axes[1].plot(history.history['loss'],
             color='#0F6E56', linewidth=2, label='Train loss')
axes[1].plot(history.history['val_loss'],
             color='#185FA5', linewidth=2,
             linestyle='--', label='Test loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss CNN par epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Graphique 3 : Comparaison finale MLP vs CNN ---
modeles  = ['MLP\nfrom scratch\n(NumPy)', 'CNN\nKeras']
accs     = [acc_mlp_cifar, acc_cnn_pct]
colors   = ['#A32D2D', '#0F6E56']
bars = axes[2].bar(modeles, accs, color=colors, alpha=0.85,
                    edgecolor='white', linewidth=0.5, width=0.4)

# Ligne du hasard
axes[2].axhline(y=10, color='gray', linewidth=1.5,
                linestyle=':', label='Hasard (10%)')

for bar, val in zip(bars, accs):
    axes[2].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=14, fontweight='bold'
    )

axes[2].set_ylabel('Accuracy TEST (%)')
axes[2].set_title('MLP vs CNN sur CIFAR-10')
axes[2].set_ylim([0, 110])
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

# Annotation de la difference
diff = acc_cnn_pct - acc_mlp_cifar
axes[2].annotate(
    f'+{diff:.1f}%\ngrace aux\nfiltres CNN',
    xy=(1, acc_cnn_pct),
    xytext=(0.5, acc_cnn_pct + 10),
    fontsize=10, color='#0F6E56', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#0F6E56')
)

plt.tight_layout()
plt.savefig('./cnn_vs_mlp_cifar10.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nGraphique sauvegarde : cnn_vs_mlp_cifar10.png")


Visualisation des predictions du CNN

In [ ]:
print("\n" + "=" * 60)
print("EXEMPLES DE PREDICTIONS DU CNN")
print("=" * 60)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(
    'Predictions du CNN sur CIFAR-10\n'
    'Vert = correct | Rouge = erreur',
    fontsize=13, fontweight='bold'
)

indices = np.random.choice(len(X_te), 32, replace=False)

for i, (ax, idx) in enumerate(zip(axes.flatten(), indices)):
    img   = X_te[idx]
    vraie  = y_test_raw[idx]
    predite = y_pred_cnn[idx]

    ax.imshow(img)
    ax.axis('off')

    couleur = '#0F6E56' if vraie == predite else '#A32D2D'
    ax.set_title(
        f'{CLASSES[predite]}',
        fontsize=7,
        color=couleur,
        fontweight='bold'
    )
    if vraie != predite:
        ax.set_xlabel(f'vrai:{CLASSES[vraie]}', fontsize=6,
                      color='gray')

plt.tight_layout()
plt.savefig('./cnn_predictions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde : cnn_predictions.png")


Conclusion

In [ ]:
print(f"""
{"=" * 60}
CONCLUSION FINALE : MLP vs CNN sur CIFAR-10
{"=" * 60}

MLP from scratch (NumPy) :
  -> Accuracy : {acc_mlp_cifar:.1f}%
  -> Raison : aplatit l'image, perd la structure spatiale
  -> Le reseau ne sait pas que les pixels voisins sont lies

CNN Keras :
  -> Accuracy : {acc_cnn_pct:.1f}%
  -> Raison : les filtres detectent bords, textures, formes
  -> Invariant aux translations (objet a gauche ou droite)

Gain du CNN : +{acc_cnn_pct - acc_mlp_cifar:.1f}% d'accuracy

Pourquoi le CNN est meilleur ?
  1. Il applique des filtres 3x3 qui glissent sur l'image
  2. Il detecte automatiquement les contours, textures, formes
  3. Le MaxPooling le rend robuste aux petits decalages
  4. Il a beaucoup moins de parametres que le MLP equivalent
     (moins d'overfitting)
{"=" * 60}
""")
